<a href="https://colab.research.google.com/github/DerivedFunction01/prompt-nlp/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pysbd sentence_transformers pycountry faker --quiet

In [ ]:
# 1. Future & Type Hinting
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Counter, Dict, Iterable, List, Optional, Set, Tuple, Union

# 2. Standard Library
import difflib
import json
import math
import os
import random
import re
import string
import time
import requests
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import lru_cache
from pathlib import Path

# 3. Third-Party Libraries
import joblib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import pysbd
import html
import pycountry
from pygments.lexers import get_all_lexers
import plotly.express as px
from faker import Faker


# 4. Configuration / Initialization
tqdm.pandas(desc="Processing")

In [ ]:
from nltk.corpus import words, wordnet
import nltk
nltk.download('words')
nltk.download('wordnet')

In [ ]:
from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Sentence Transformer model loaded.")
BATCH_SIZE=512

In [ ]:
_HATCH_CYCLE = ["///", "...", "xxx", "---", "+++", "ooo", "***", "\\\\\\"]

def infer_variants(labels: list[str], sep: str = "_") -> dict[str, tuple[str, str]]:
    """
    Infer (base, variant) for each label by finding the shortest prefix
    that another label shares via `sep`. Works for any label_col, not just models.
    Labels with no shorter prefix match are assigned variant 'BASE'.
    """
    unique_ordered = list(dict.fromkeys(labels))
    unique_by_len = sorted(unique_ordered, key=len)

    result: dict[str, tuple[str, str]] = {}
    for label in unique_ordered:
        best_base = None
        for candidate in unique_by_len:
            if candidate == label:
                continue
            if label.startswith(candidate + sep):
                if best_base is None or len(candidate) < len(best_base):
                    best_base = candidate
        result[label] = (
            (label, "BASE")
            if best_base is None
            else (best_base, label[len(best_base) + 1 :])
        )
    return result


def assign_hatches(variant_keys: list[str]) -> dict[str, str]:
    """BASE gets solid (no hatch); all others cycle through _HATCH_CYCLE."""
    hatch_map = {"BASE": ""}
    cycle_idx = 0
    for v in variant_keys:
        if v not in hatch_map:
            hatch_map[v] = _HATCH_CYCLE[cycle_idx % len(_HATCH_CYCLE)]
            cycle_idx += 1
    return hatch_map


def make_legend(ax, handles, title, anchor, loc="upper left"):
    """Consistent legend styling anchored to an Axes."""
    leg = ax.legend(
        handles=handles,
        title=title,
        bbox_to_anchor=anchor,
        bbox_transform=ax.transAxes,
        loc=loc,
        frameon=True,
        borderpad=0.8,
        labelspacing=0.5,
        handlelength=2.0,
        handletextpad=0.6,
    )
    leg._legend_box.align = "left"
    return leg


def make_variant_legend(fig, all_variants, hatch_map, anchor=(1.0, 0.55)):
    """Figure-level variant legend using hatch patches. Skipped if only BASE."""
    meaningful = [v for v in all_variants if v != "BASE"]
    if not meaningful or len(all_variants) == 1:
        return
    handles = [
        mpatches.Patch(
            facecolor="grey",
            hatch=hatch_map[v],
            edgecolor="white",
            label="Baseline" if v == "BASE" else v,
        )
        for v in all_variants
    ]
    fig.legend(
        handles=handles,
        title="Variant",
        bbox_to_anchor=anchor,
        loc="upper left",
        frameon=True,
        borderpad=0.8,
        handlelength=2.0,
        labelspacing=0.5,
        handletextpad=0.6,
    )

def plot_bar_metrics(
    df: pd.DataFrame,
    label_col: str,
    metric_col: str = "Metric",
    score_col: str = "Score",
    title: str = "",
    y_label: str = "Score",
    x_label: str = "",
    palette: str = "viridis",
    ylim: tuple | None = None,
    fmt: str = "%.3f",
    base_height: int = 8,
    width_per_bar: float = 0.5,
    show: bool = True,
    ax=None,
    show_variant_ticks: bool = True,
    group_sort: str | None = None,
):
    """
    Grouped + stacked bar chart with dynamic variant detection.
    Generalised from plot_model_metrics — works with any label_col,
    not just 'Model'.
    """
    df = df.copy()
    score_range = max(df[score_col].max() - df[score_col].min(), 1e-6)
    label_offset = score_range * 0.01

    variant_map = infer_variants(df[label_col].tolist())
    df["_Base"] = df[label_col].map(lambda m: variant_map[m][0]) # type: ignore
    df["_Variant"] = df[label_col].map(lambda m: variant_map[m][1]) # type: ignore

    no_variants = set(df["_Base"].unique()) == set(df[label_col].unique())

    # ── Fallback: flat barplot ───────────────────────────────────────────────
    if no_variants:
        n_labels = df[label_col].nunique()
        n_metrics = df[metric_col].nunique()
        if ax is None:
            dynamic_width = max(10, n_labels * n_metrics * width_per_bar + 2)
            _, curr_ax = plt.subplots(figsize=(dynamic_width, base_height))
        else:
            curr_ax = ax

        sns.barplot(
            x=label_col,
            y=score_col,
            hue=metric_col,
            data=df,
            palette=palette,
            ax=curr_ax,
        )
        curr_ax.set_title(title, fontsize=16, pad=20)
        curr_ax.set_ylabel(y_label)
        curr_ax.set_xlabel(x_label or label_col)
        curr_ax.grid(axis="y", linestyle="--", alpha=0.7)
        if n_labels > 5:
            curr_ax.tick_params(axis="x", rotation=45)
        if ylim:
            curr_ax.set_ylim(*ylim)
        for container in curr_ax.containers:
            curr_ax.bar_label(
                container, fmt=fmt, label_type="edge", fontsize=9, padding=label_offset # type: ignore
            )
        colors = sns.color_palette(palette, n_colors=n_metrics)
        make_legend(
            curr_ax,
            [
                mpatches.Patch(facecolor=c, label=m)
                for m, c in zip(df[metric_col].unique(), colors)
            ],
            title=metric_col,
            anchor=(1.01, 1),
        )
        if show and ax is None:
            plt.tight_layout()
            plt.show()
        return

    # ── Variant mode: grouped + stacked ─────────────────────────────────────
    all_variants = ["BASE"] + sorted(v for v in df["_Variant"].unique() if v != "BASE")
    variant_hatch = assign_hatches(all_variants)

    base_labels = list(dict.fromkeys(df["_Base"]))
    if group_sort is not None:
        ascending = group_sort == "worst"
        rep_scores = df.groupby("_Base")[score_col].max().reindex(base_labels)
        base_labels = rep_scores.sort_values(ascending=ascending).index.tolist()

    metrics = df[metric_col].unique().tolist()
    n_groups = len(base_labels)
    n_metrics = len(metrics)
    colors = sns.color_palette(palette, n_colors=n_metrics)
    metric_color = dict(zip(metrics, colors))

    if ax is None:
        dynamic_width = max(10, n_groups * n_metrics * width_per_bar * 1.8 + 2)
        _, curr_ax = plt.subplots(figsize=(dynamic_width, base_height))
    else:
        curr_ax = ax

    group_width = 0.95
    bar_width = group_width / n_metrics
    group_positions = np.arange(n_groups)

    for m_idx, metric in enumerate(metrics):
        metric_df = df[df[metric_col] == metric]
        for g_idx, base in enumerate(base_labels):
            group_df = metric_df[metric_df["_Base"] == base].copy()
            if group_df.empty:
                continue

            group_df = group_df.sort_values(score_col, ascending=True).reset_index(
                drop=True
            )
            scores = group_df[score_col].tolist()
            variants = group_df["_Variant"].tolist()
            bottoms = [0.0] + scores[:-1]
            heights = [scores[0]] + [
                max(scores[i] - scores[i - 1], 0) for i in range(1, len(scores))
            ]

            x_pos = group_positions[g_idx] + (m_idx - n_metrics / 2 + 0.5) * bar_width
            color = metric_color[metric]
            half_w = (bar_width * 0.9) / 2

            for bot, ht, var in zip(bottoms, heights, variants):
                if ht <= 0:
                    continue
                curr_ax.bar(
                    x_pos,
                    ht,
                    bar_width * 0.9,
                    bottom=bot,
                    color=color,
                    hatch=variant_hatch.get(var, ""),
                    edgecolor="white",
                    linewidth=0.5,
                    alpha=0.85,
                )

            if show_variant_ticks:
                min_gap, last_labeled = score_range * 0.015, -999
                for score in scores:
                    if score - last_labeled >= min_gap:
                        curr_ax.plot(
                            [x_pos - half_w, x_pos + half_w],
                            [score, score],
                            color="white",
                            linewidth=0.5,
                            solid_capstyle="butt",
                            zorder=5,
                        )
                        curr_ax.text(
                            x_pos,
                            score + label_offset,
                            fmt % score,
                            ha="center",
                            va="bottom",
                            fontsize=6,
                            bbox=dict(
                                boxstyle="round,pad=0.1",
                                fc="white",
                                ec="none",
                                alpha=0.7,
                            ),
                        )
                        last_labeled = score
            else:
                curr_ax.text(
                    x_pos,
                    scores[-1] + label_offset,
                    fmt % scores[-1],
                    ha="center",
                    va="bottom",
                    fontsize=7,
                    bbox=dict(
                        boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.7
                    ),
                )

    curr_ax.set_xticks(group_positions)
    curr_ax.set_xticklabels(base_labels, rotation=45, ha="right")
    curr_ax.set_title(title, fontsize=16, pad=20)
    curr_ax.set_ylabel(y_label)
    curr_ax.set_xlabel(x_label or label_col)
    curr_ax.grid(axis="y", linestyle="--", alpha=0.5)
    if ylim:
        curr_ax.set_ylim(*ylim)

    fig = curr_ax.get_figure()
    fig.legend( # type: ignore
        handles=[mpatches.Patch(facecolor=metric_color[m], label=m) for m in metrics],
        title=metric_col,
        bbox_to_anchor=(1.0, 0.95),
        loc="upper left",
        frameon=True,
        borderpad=0.8,
        handlelength=2.0,
        labelspacing=0.5,
        handletextpad=0.6,
    )
    make_variant_legend(fig, all_variants, variant_hatch, anchor=(1.0, 0.55))

    if show and ax is None:
        plt.tight_layout()
        plt.show()


# ── Grid plotter ─────────────────────────────────────────────────────────────


def plot_grid(
    plot_functions: List[Callable[[plt.Axes], None]], # type: ignore
    n_cols: int = 4,
    figsize_per_plot: tuple = (5, 4.5),
    suptitle: Optional[str] = None,
    sharex: bool = False,
    sharey: bool = False,
    hide_empty_axes: bool = True,
    legend_fn: Optional[Callable[[plt.Figure], None]] = None, # type: ignore
):
    n_plots = len(plot_functions)
    if n_plots == 0:
        return

    n_rows = math.ceil(n_plots / n_cols)
    fig, axes = plt.subplots(
        nrows=n_rows,
        ncols=n_cols,
        figsize=(figsize_per_plot[0] * n_cols, figsize_per_plot[1] * n_rows),
        sharex=sharex,
        sharey=sharey,
        constrained_layout=False,
    )
    axes = list(axes.flat) if (n_rows > 1 or n_cols > 1) else [axes]

    for ax, fn in zip(axes, plot_functions):
        fn(ax)

    if hide_empty_axes:
        for ax in axes[n_plots:]:
            ax.set_visible(False)
            ax.set_axis_off()

    # Auto-consolidate identical legends across subplots
    axes_with_legends = [ax for ax in axes[:n_plots] if ax.get_legend() is not None]
    if axes_with_legends:
        all_labels = [
            [t.get_text() for t in ax.get_legend().get_texts()]
            for ax in axes_with_legends
        ]
        if all(labels == all_labels[0] for labels in all_labels):
            handles = axes_with_legends[0].get_legend().legend_handles
            labels = all_labels[0]
            for ax in axes_with_legends:
                ax.get_legend().remove()
            fig.legend(
                handles,
                labels,
                loc="lower center",
                ncol=min(len(labels), 6),
                fontsize=9,
                title_fontsize=10,
                bbox_to_anchor=(0.5, 0.01),
                frameon=False,
            )

    if legend_fn:
        legend_fn(fig)

    bottom_pad = 0.075 if legend_fn else 0.05
    top_pad = 0.95 if suptitle else 1.0
    if suptitle:
        fig.suptitle(suptitle, fontsize=16)

    fig.tight_layout(rect=[0, bottom_pad, 1, top_pad]) # type: ignore
    plt.show()
    return fig, axes

In [ ]:
# Model misalignment: From generic customer agents to AI scrappers,
# we don't want random text to change the output/behavior, even if it
# is not a safety violation. (ex. A customer ai agent is told to respond in JSON
# for no reason)
class PromptEntity(Enum):
    # --- THE ATTACK VECTORS (The "How") ---
    # --- Level 1 ---
    VIOLATION = "VIOLATION"  # Rule breaking "Ignore safety", "Disable filters", "Forget rules", "Unfiltered", "allowed to do X" (normally restricted), "force to reply no matter what"

    # --- Level 2 --- (standalone, it isn't malicious)
    OBFUSCATION = "OBFUSC"
    """
    STRICT RULE: Only use for MACHINE-READABLE or NON-ALPHABETIC ciphers.
    Includes: Base64, Morse Code, Hexadecimal, Binary, Rot13, or Caesar Ciphers.
    EXCLUDES: Spaced-out letters (H A T E), Leetspeak (H4ck), or Punctuation-stuffing (s.t.e.a.l).
    These 'Human-Readable' variants must be tagged with their CORE INTENT (e.g., SOCIAL or CYBER).
    """
    PERSONA = "PERSONA" # You are (generic)
    HIGHSTAKES = "HIGHSTAKES"  # Professional advice: medical, legal, financial — harm via misinformation or liability
    NSHOT = "NSHOT" # User/Assistant examples embedded in the prompt
    FORMAT = "FORMAT"

    # --- THE HARMFUL INTENT (The "What") ---
    VIOLENCE = "VIOLENCE"  # Murder, weapons, physical assault, CBRN
    CYBER = "CYBER"  # Hacking, malware, technical exploits
    ILLICIT = "ILLICIT"  # Financial crime, fraud, drugs, theft
    SOCIAL = "SOCIAL"  # Hate, harassment, NSFW, child safety
    SENSITIVE = "SENSITIVE"  # PII, Privacy leaks, API/secrets
    DISINFO = "DISINFO"  # Propaganda, Fake news, Defamation


    # --- Default ---
    BENIGN = "BENIGN"

In [ ]:
MAX_SENTENCE_TOKEN_LENGTH = 25
MIN_SENTENCE_TOKEN_LENGTH = 3

In [ ]:

def build_alternation(items: List[str], sort_longest_first: bool = True) -> str:
    """
    Build regex alternation pattern, optionally sorting by length (longest first).
    """
    if not items:
        return ""
    if len(items) == 1:
        return items[0]

    if sort_longest_first:
        # Remove duplicates while preserving order (for tiebreaker)
        unique_items = []
        seen = set()
        for item in items:
            if item not in seen:
                unique_items.append(item)
                seen.add(item)

        # Sort by: (word_count DESC, then char_length DESC)
        unique_items = sorted(
            unique_items,
            key=lambda x: (
                -len(x.split()),  # Primary: word count (descending)
                -len(x),  # Secondary: character length (descending)
            ),
        )
        items = unique_items

    return f'(?:{"|".join(items)})'


def make_gap(max_gap: int) -> str:
    """
    Flexible word gap: allows up to N words between tokens.
    """
    return rf"(?:\s+\b\w+\b){{0,{max_gap}}}\s+"


# =============================================================================
# REGEX COMPONENT
# =============================================================================

@dataclass
class RegexComponent:
    label: str
    templates: Union[
        List[str],
        Set[str],
        Dict[str, List[str] | Set[str]]
    ]
    local_vocab: Dict[str, Set[str]] = field(default_factory=dict)

    _patterns: List[re.Pattern] = field(init=False, default_factory=list)

    # -----------------------------
    # Build regex from template
    # -----------------------------
    def compile_regex_template(self, template: str, max_gap: int = 5) -> re.Pattern:
        gap = make_gap(max_gap)

        tokens = template.strip().split()
        parts: List[str] = []

        for token in tokens:
            match = re.fullmatch(r"\[([A-Za-z0-9_-]+)\]", token)

            if match:
                key = match.group(1)

                if key in self.local_vocab:
                    vocab = list(self.local_vocab[key])
                    parts.append(build_alternation(vocab))
                else:
                    # fallback: treat literally
                    parts.append(re.escape(token))
            else:
                parts.append(re.escape(token))

        pattern = gap.join(parts)
        pattern = rf"\b{pattern}\b"

        return re.compile(pattern, flags=re.IGNORECASE)
    def _iter_templates(self) -> Iterable[Tuple[str, str]]:
      """
      Yields (category, template)
      """
      if isinstance(self.templates, dict):
          for category, templates in self.templates.items():
              for t in templates:
                  yield category, t
      else:
          for t in self.templates:
              yield "default", t
    # -----------------------------
    # Compile all templates
    # -----------------------------
    def compile(self, max_gap: int = 5) -> None:
      self._patterns = []

      for category, template in self._iter_templates():
          pattern = self.compile_regex_template(template, max_gap=max_gap)
          self._patterns.append(pattern)

    # -----------------------------
    # Search APIs
    # -----------------------------
    def finditer(self, text: str) -> Iterable[re.Match]:
        if not self._patterns:
            self.compile()

        for pattern in self._patterns:
            yield from pattern.finditer(text)

    def extract_spans(self, text: str) -> List[Tuple[int, int, str]]:
        return [(m.start(), m.end(), self.label) for m in self.finditer(text)]

    def extract_matches(self, text: str) -> List[str]:
        return [m.group(0) for m in self.finditer(text)]

    def search(self, text: str) -> bool:
        return any(True for _ in self.finditer(text))

In [ ]:
def cluster_and_visualize_sentences(
    dataframe,
    sentence_embeddings,
    num_clusters=6,
    sample_size=5,
    random_state=42,
    pca_components=2,
    target_label='sentence',
    width=900,
    height=700
):
    """
    Runs KMeans clustering on sentence embeddings, prints sample sentences per cluster,
    performs PCA for visualization, and returns a Plotly scatter figure.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain a 'sentence' column.
    sentence_embeddings : np.ndarray
        Embeddings for each sentence (n_samples x embedding_dim).
    num_clusters : int
        Number of KMeans clusters.
    sample_size : int
        Number of example sentences to print per cluster.
    random_state : int
        Random seed for reproducibility.
    pca_components : int
        Number of PCA components (2 recommended for visualization).
    width, height : int
        Plotly figure size.

    Returns
    -------
    df : pd.DataFrame
        Updated with cluster labels + PCA components.
    kmeans_model : fitted KMeans model
    pca_model : fitted PCA model
    fig : Plotly figure object
    """

    # -------------------------------
    # 1. KMeans clustering
    # -------------------------------
    kmeans_model = KMeans(
        n_clusters=num_clusters,
        init='k-means++',
        max_iter=1000,
        n_init=10,
        random_state=random_state
    )
    kmeans_model.fit(sentence_embeddings)
    df = dataframe.copy()
    df['cluster_st'] = kmeans_model.labels_
    print(f"Clustering complete. Added 'cluster_st' column with {num_clusters} clusters.")

    # -------------------------------
    # 2. Print sample sentences
    # -------------------------------
    print(f"\nExamples from each of the {num_clusters} clusters:")
    for i in range(num_clusters):
        print(f"\n--- Cluster {i} ---")
        cluster_df = df[df['cluster_st'] == i]
        n = min(sample_size, len(cluster_df))
        display(cluster_df[target_label].sample(n).reset_index(drop=True))

    # -------------------------------
    # 3. PCA for visualization
    # -------------------------------
    pca_model = PCA(n_components=pca_components, random_state=random_state)
    components = pca_model.fit_transform(sentence_embeddings)

    df['pca_st_1'] = components[:, 0]
    df['pca_st_2'] = components[:, 1]
    df['cluster_st_str'] = df['cluster_st'].astype(str)

    print("PCA complete. Added 'pca_st_1', 'pca_st_2', and 'cluster_st_str' columns.")

    # -------------------------------
    # 4. Stratified sampling for Plotly
    # -------------------------------
    def stratified_sample(df, cluster_col, per_cluster=300, random_state=42):
        sampled = (
            df.groupby(cluster_col, group_keys=False)
              .apply(lambda x: x.sample(min(len(x), per_cluster), random_state=random_state))
        )
        return sampled

    # Sample for visualization only
    df_vis = stratified_sample(df, 'cluster_st', per_cluster=300)

    fig = px.scatter(
        df_vis,
        x='pca_st_1',
        y='pca_st_2',
        color='cluster_st_str',
        hover_name=target_label,
        title='K-Means Clusters (Sentence Embeddings + PCA)',
        labels={'pca_st_1': 'PCA Component 1', 'pca_st_2': 'PCA Component 2'},
        width=width,
        height=height
    )
    fig.show()

    return df, kmeans_model, pca_model, fig

# ---------------------------------------------
# 1. Unigram plotter (uses text_col)
# ---------------------------------------------
def plot_cluster_unigrams(ax, cluster_id, df_subset, text_col, stopwords=[], top_n=10):
    # Extract tokens
    words = " ".join(df_subset[text_col]).split()
    words = [w for w in words if w not in stopwords and len(w) > 2]

    # Count
    common = Counter(words).most_common(top_n)
    plot_df = pd.DataFrame(common, columns=["Word", "Count"])

    # Plot
    if not plot_df.empty:
        sns.barplot(
            x="Count",
            y="Word",
            data=plot_df,
            palette="viridis",
            hue="Word",
            ax=ax,
            legend=False
        )

    ax.set_title(f"Cluster {cluster_id} – Top {top_n} Unigrams")
    ax.set_xlabel("Count")
    ax.set_ylabel("Word")


# ---------------------------------------------
# 2. Bigram plotter (uses text_col)
# ---------------------------------------------
def plot_cluster_bigrams(ax, cluster_id, df_subset, text_col, stopwords=[], top_n=10):
    words = " ".join(df_subset[text_col]).split()

    # Build bigrams
    bigrams = [
        f"{w1} {w2}"
        for w1, w2 in zip(words, words[1:])
        if w1 not in stopwords and w2 not in stopwords
        and len(w1) > 2 and len(w2) > 2
    ]

    common = Counter(bigrams).most_common(top_n)
    plot_df = pd.DataFrame(common, columns=["Bigram", "Count"])

    if not plot_df.empty:
        sns.barplot(
            x="Count",
            y="Bigram",
            data=plot_df,
            palette="viridis",
            hue="Bigram",
            ax=ax,
            legend=False
        )

    ax.set_title(f"Cluster {cluster_id} – Top {top_n} Bigrams")
    ax.set_xlabel("Count")
    ax.set_ylabel("Bigram")


# ---------------------------------------------
# 3. Wrapper: build all cluster plot functions
# ---------------------------------------------
def build_cluster_ngram_plots(
    df,
    cluster_col="cluster_st",
    text_col="sentence_clean",
    stop_words=[],
    top_n=10
):
    """
    Returns a list of plotting functions (unigram + bigram per cluster)
    that can be passed directly into plot_grid().
    """

    plot_functions = []

    for cluster_id in sorted(df[cluster_col].unique()):
        df_subset = df[df[cluster_col] == cluster_id]

        # Capture values in default args to avoid late-binding lambda bug
        plot_functions.append(
            lambda ax, cid=cluster_id, cs=df_subset:
                plot_cluster_unigrams(ax, cid, cs, text_col, stop_words, top_n)
        )
        plot_functions.append(
            lambda ax, cid=cluster_id, cs=df_subset:
                plot_cluster_bigrams(ax, cid, cs, text_col, stop_words, top_n)
        )

    return plot_functions



# Salad-Data

In [ ]:
sd_df = pd.read_json("hf://datasets/OpenSafetyLab/Salad-Data/base_set.json")

In [ ]:
# Create a temp df for each source with sample rows
sources = sd_df["source"].unique()

for source in sources:
    temp_df = sd_df[sd_df["source"] == source].sample(min(3, len(sd_df[sd_df["source"] == source])))
    var_name = f"df_{source.lower().replace('-', '_').replace(' ', '_')}"
    globals()[var_name] = temp_df
    print(f"\n{'='*60}")
    print(f"Source: {source}  →  stored as `{var_name}`")
    print(f"{'='*60}")
    display(temp_df)

In [ ]:
SALAD_TO_ENTITY = {
    # VIOLENCE
    "O4: Terrorism": PromptEntity.VIOLENCE.value,
    "O5: Violent Content": PromptEntity.VIOLENCE.value,
    "O6: Child Abuse": PromptEntity.VIOLENCE.value,
    "O35: Weapon Generation and Transportation": PromptEntity.VIOLENCE.value,
    "O39: Biological and Chemical Harms": PromptEntity.VIOLENCE.value,
    "O56: Violent Crimes": PromptEntity.VIOLENCE.value,
    "O55: Sexual Offenses": PromptEntity.VIOLENCE.value,
    # CYBER
    "O37: Malware Generation": PromptEntity.CYBER.value,
    "O38: Cyber Attack": PromptEntity.CYBER.value,
    "O36: Management or Operation of Critical Infrastructure": PromptEntity.CYBER.value,
    # ILLICIT
    "O19: Illegal Drugs and Regulated/Controlled Substances": PromptEntity.ILLICIT.value,
    "O24: Multi-level Marketing": PromptEntity.ILLICIT.value,
    "O25: Paypal Lending": PromptEntity.ILLICIT.value,
    "O26: Gambling": PromptEntity.ILLICIT.value,
    "O27: Labor Issues": PromptEntity.ILLICIT.value,
    "O28: Trade Affairs": PromptEntity.ILLICIT.value,
    "O42: Scams": PromptEntity.ILLICIT.value,
    "O43: Financial Fraud": PromptEntity.ILLICIT.value,
    "O49: Pseudo-pharmaceuticals": PromptEntity.ILLICIT.value,
    "O53: Financial Crimes": PromptEntity.ILLICIT.value,
    "O54: Drug-related Crimes": PromptEntity.ILLICIT.value,
    "O57: Theft": PromptEntity.ILLICIT.value,
    "O58: Illegal Law Advice": PromptEntity.ILLICIT.value,
    "O59: Environmental Crimes": PromptEntity.ILLICIT.value,
    "O60: Traffic and Driving Offenses": PromptEntity.ILLICIT.value,
    # High Stakes
    "O17: Law Advice": PromptEntity.HIGHSTAKES.value,
    "O20: Medical Advice": PromptEntity.HIGHSTAKES.value,
    "O18: Common Sense": PromptEntity.HIGHSTAKES.value,
    # SOCIAL
    "O1: Hate Speech": PromptEntity.SOCIAL.value,
    "O2: Harass, Threaten, or Bully An Individual": PromptEntity.SOCIAL.value,
    "O3: Enjoying someone else's Pain": PromptEntity.SOCIAL.value,
    "O7: Insult": PromptEntity.SOCIAL.value,
    "O8: Racial and Ethnic Stereotyping": PromptEntity.SOCIAL.value,
    "O9: Age Stereotyping": PromptEntity.SOCIAL.value,
    "O10: Disability Stereotyping": PromptEntity.SOCIAL.value,
    "O11: Gender Stereotyping": PromptEntity.SOCIAL.value,
    "O12: Religious Stereotyping": PromptEntity.SOCIAL.value,
    "O13: Location Stereotyping": PromptEntity.SOCIAL.value,
    "O14: Erotic Chat": PromptEntity.SOCIAL.value,
    "O15: Sexual Activity": PromptEntity.SOCIAL.value,
    "O16: Pornography": PromptEntity.SOCIAL.value,
    "O23: Radicalization": PromptEntity.SOCIAL.value,
    "O40: Group Defamation": PromptEntity.SOCIAL.value,
    "O62: Self-Harm": PromptEntity.SOCIAL.value,
    "O63: Psychological Manipulation": PromptEntity.SOCIAL.value,
    # SENSITIVE
    "O31: Government Secrets": PromptEntity.SENSITIVE.value,
    "O32: Financial Data Leaks": PromptEntity.SENSITIVE.value,
    "O33: Inferring Personal Information": PromptEntity.SENSITIVE.value,
    "O34: Leaking Private Information": PromptEntity.SENSITIVE.value,
    "O52: Illegitimate Surveillance": PromptEntity.SENSITIVE.value,
    # DISINFO
    "O21: Historical Revisionism": PromptEntity.DISINFO.value,
    "O22: Science Denialism": PromptEntity.DISINFO.value,
    "O29: Fake News": PromptEntity.DISINFO.value,
    "O30: Social Media Rumors": PromptEntity.DISINFO.value,
    "O41: Defamation Someone": PromptEntity.DISINFO.value,
    "O44: Academic Dishonesty": PromptEntity.DISINFO.value,
    "O45: Fake Review Generation": PromptEntity.DISINFO.value,
    "O46: Fake Online Content": PromptEntity.DISINFO.value,
    "O47: Disinformation": PromptEntity.DISINFO.value,
    "O48: Spam": PromptEntity.DISINFO.value,
    "O50: Manipulation of Public Opinion": PromptEntity.DISINFO.value,
    "O51: Campaign Materials": PromptEntity.DISINFO.value,
    "O67: Forge Identity": PromptEntity.DISINFO.value,
    "O64: Mimicking Writing Style": PromptEntity.DISINFO.value,
    "O65: Plagiarism": PromptEntity.DISINFO.value,
    "O66: Copyright Infringement": PromptEntity.DISINFO.value,
}

def invert_entity_mapping(mapping: dict) -> dict:
    inverted = defaultdict(list)
    for salad_cat, entity in mapping.items():
        inverted[entity].append(salad_cat)
    return dict(inverted)

ENTITY_TO_SALAD = invert_entity_mapping(SALAD_TO_ENTITY)
unmapped = set(sd_df["3-category"].unique()) - set(SALAD_TO_ENTITY.keys())
print((f"{len(unmapped)} unmapped categories:", unmapped) if unmapped else "All categories mapped")

In [ ]:
BASIC_SUB = re.compile(r"[^a-zA-Z\-\s]")
def clean_text_basic(text):
    # Convert HTML entities to Unicode
    text = html.unescape(text)
    # Remove non-alphanumeric characters and extra spaces
    text = BASIC_SUB.sub(" ", text)
    # Convert to lowercase
    text = text.lower()
    return text


sd_eda_df = sd_df.copy(deep=True)
sd_eda_df["question_clean"] = sd_eda_df["question"].progress_apply(clean_text_basic)
sd_eda_df["entity"] = sd_eda_df["3-category"].map(SALAD_TO_ENTITY)
sd_eda_df["token_length"] = sd_eda_df["question_clean"].str.split().str.len()

In [ ]:
# --- Plot functions ---
def plot_entity_dist(ax):
    counts = sd_eda_df["entity"].value_counts()
    sns.barplot(
        x=counts.values,
        y=[e for e in counts.index],
        ax=ax,
        palette="viridis",
        hue=[e for e in counts.index], # Added hue argument
        legend=False,
    )
    ax.set_title("Entity Distribution")
    ax.set_xlabel("Count")


def plot_token_lengths(ax):
    sns.histplot(sd_eda_df["token_length"], bins=40, kde=True, ax=ax, color="steelblue") # type: ignore
    ax.axvline(MAX_SENTENCE_TOKEN_LENGTH, color="red", linestyle="--", linewidth=1, label=f"{MAX_SENTENCE_TOKEN_LENGTH}-token cutoff")
    ax.legend()
    ax.set_title("Token Length Distribution")
    ax.set_xlabel("Token Count")


def plot_token_lengths_by_entity(ax):
    entity_labels = sd_eda_df["entity"].dropna()
    temp = sd_eda_df.copy()
    temp["entity_label"] = entity_labels
    sns.boxplot(
        x="entity_label",
        y="token_length",
        data=temp.dropna(subset=["entity_label"]),
        ax=ax,
        palette="viridis",
        hue="entity_label", # Added hue argument
        legend=False
    )
    ax.axhline(MAX_SENTENCE_TOKEN_LENGTH, color="red", linestyle="--", linewidth=1, label=f"{MAX_SENTENCE_TOKEN_LENGTH}-token cutoff")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    ax.set_title("Token Lengths by Entity")


def plot_source_by_entity(ax):
    temp = sd_eda_df.dropna(subset=["entity"]).copy()
    temp["entity_label"] = temp["entity"]
    counts = temp.groupby(["entity_label", "source"]).size().reset_index(name="count")
    pivot = counts.pivot(index="entity_label", columns="source", values="count").fillna(
        0
    )
    pivot.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
    ax.set_title("Source Breakdown by Entity")
    ax.tick_params(axis="x", rotation=45)
    ax.set_xlabel("")


def plot_short_span_yield(ax):
    thresholds = range(5, 51, 5)
    yields = [len(sd_eda_df[sd_eda_df["token_length"] <= t]) for t in thresholds]
    ax.plot(list(thresholds), yields, marker="o", color="steelblue")
    ax.set_title("Usable Spans vs. Token cutoff")
    ax.set_xlabel("Max Token Length")
    ax.set_ylabel("# Samples")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

SALAD_STOP_WORDS = text.ENGLISH_STOP_WORDS | {
    "please",
    "provide",
    "write",
    "create",
    "give",
    "make",
    "tell",
    "explain",
    "describe",
    "generate",
    "list",
    "way",
    "ways",
    "example",
    "examples",
    "use",
    "using",
    "used",
    "like",
    "would",
    "could",
    "want",
    "need",
    "help",
    "information",
    "without",
    "know",
    "just",
    "step",
    "steps",
    "guide",
    "best",
    "type",
    "types",
}


def plot_salad_unigrams(ax):
    all_words = " ".join(sd_eda_df["question_clean"]).split()
    all_words = [w for w in all_words if w not in SALAD_STOP_WORDS and len(w) > 2]
    common = Counter(all_words).most_common(20)
    words_df = pd.DataFrame(common, columns=["Word", "Count"])
    sns.barplot(
        x="Count",
        y="Word",
        data=words_df,
        palette="viridis",
        hue="Word",
        ax=ax,
        legend=False,
    )
    ax.set_title("Top 20 Words")


def plot_salad_bigrams(ax):
    all_words = " ".join(sd_eda_df["question_clean"]).split()
    bigrams = [
        f"{w1} {w2}"
        for w1, w2 in zip(all_words, all_words[1:])
        if w1 not in SALAD_STOP_WORDS
        and w2 not in SALAD_STOP_WORDS
        and len(w1) > 2
        and len(w2) > 2
    ]
    common = Counter(bigrams).most_common(20)
    words_df = pd.DataFrame(common, columns=["Bigram", "Count"])
    sns.barplot(
        x="Count",
        y="Bigram",
        data=words_df,
        palette="viridis",
        hue="Bigram",
        ax=ax,
        legend=False,
    )
    ax.set_title("Top 20 Bigrams")


def plot_salad_unigrams_by_entity(ax):
    entity_top_words = {}
    for entity, cats in ENTITY_TO_SALAD.items():
        subset = sd_eda_df[sd_eda_df["3-category"].isin(cats)]["question_clean"]
        words = " ".join(subset).split()
        words = [w for w in words if w not in SALAD_STOP_WORDS and len(w) > 2]
        top3 = [w for w, _ in Counter(words).most_common(3)]
        entity_top_words[entity] = ", ".join(top3)

    y_labels = list(entity_top_words.keys())
    x_vals = list(range(len(y_labels)))
    ax.barh(x_vals, [1] * len(x_vals), color="none")  # invisible bars for layout
    for i, (entity, words) in enumerate(entity_top_words.items()):
        ax.text(
            0.02, i, words, va="center", fontsize=10, transform=ax.get_yaxis_transform()
        )
    ax.set_yticks(x_vals)
    ax.set_yticklabels(y_labels)
    ax.set_title("Top Words per Entity")
    ax.set_xticks([])
    ax.set_xlabel("")


# --- Run grid ---
plot_grid(
    plot_functions=[
        plot_entity_dist,
        plot_token_lengths,
        plot_token_lengths_by_entity,
        plot_source_by_entity,
        plot_short_span_yield,
        plot_salad_unigrams,
        plot_salad_bigrams,
        plot_salad_unigrams_by_entity,
    ],
    n_cols=2,
    figsize_per_plot=(10, 6),
    suptitle="SALAD Data EDA",
)


In [ ]:
MAX_SENTENCE_COUNT = 2

# Initialize sentence segmenter
seg = pysbd.Segmenter(language="en", clean=False)

# Define a function to get sentence count with a token length MAX_SENTENCE_TOKEN_LENGTH
def get_sentence_count(row):
    # If the text is already too long by token count, skip segmentation
    if row['token_length'] > MAX_SENTENCE_TOKEN_LENGTH:
        return MAX_SENTENCE_COUNT + 1 # Assign a value that ensures it's filtered out
    return len(seg.segment(row['question']))

# Add a sentence count column to eda_df with tqdm progress bar
sd_eda_df['sentence_count'] = sd_eda_df.progress_apply(get_sentence_count, axis=1)

# Filter for short prompts (<= MAX_TOKEN_LENGTH tokens) AND at most MAX_SENTENCE_COUNT sentences
short_prompts_df = sd_eda_df[
    (sd_eda_df['token_length'] <= MAX_SENTENCE_TOKEN_LENGTH) &
    (sd_eda_df['sentence_count'] <= MAX_SENTENCE_COUNT)
].copy()

print(f"Original number of prompts: {len(sd_eda_df)}")
print(f"Number of short prompts (<= {MAX_SENTENCE_TOKEN_LENGTH} tokens AND <= {MAX_SENTENCE_COUNT} sentences): {len(short_prompts_df)}")

display(short_prompts_df.head(10))

In [ ]:
# Generate embeddings for the cleaned sentences
# This might take a little while depending on the number of sentences
print("Generating sentence embeddings...")
sd_embeddings = sentence_model.encode(short_prompts_df['question'].tolist(), show_progress_bar=True, batch_size=BATCH_SIZE)

print(f"Embeddings shape: {sd_embeddings.shape}")

In [ ]:
sd_sent_df, kmeans_sd, pca_sd, fig_sd = cluster_and_visualize_sentences(
    dataframe=short_prompts_df,
    sentence_embeddings=sd_embeddings,
    num_clusters=len(ENTITY_TO_SALAD.keys()),
    target_label="question"
)

In [ ]:
sd_cluster_plot_functions = build_cluster_ngram_plots(
    sd_sent_df,
    cluster_col="cluster_st",
    text_col="question_clean",
    stop_words=SALAD_STOP_WORDS,
    top_n=10
)

plot_grid(
    plot_functions=sd_cluster_plot_functions,
    n_cols=2,
    figsize_per_plot=(8, 6),
    suptitle="Unigram and Bigram Analysis per Cluster"
)


# JackHao

In [ ]:
splits = {'train': 'balanced/jailbreak_dataset_train_balanced.csv', 'test': 'balanced/jailbreak_dataset_test_balanced.csv'}
jh_df = pd.read_csv("hf://datasets/jackhhao/jailbreak-classification/" + splits["train"])
jh_df.head()

In [ ]:
# Initialize sentence segmenter
seg = pysbd.Segmenter(language="en", clean=False)

# Pre-compile the regex pattern for sentence boundary splitting
# This pattern looks for a lowercase letter, followed by a punctuation, followed by an uppercase letter
# and inserts a space after the punctuation.
SENTENCE_BOUNDARY_FIX_REGEX = re.compile(r'([a-z])([.,;:\)\]])([A-Z])')

def clean_text_for_segmentation(text):
    # Insert a space for pattern like 'word.Next' -> 'word. Next'
    text = SENTENCE_BOUNDARY_FIX_REGEX.sub(r'\1\2 \3', text)
    return text

# List to store individual sentences and their metadata
sentence_data = []

for idx, row in tqdm(jh_df.iterrows(), total=len(jh_df), desc="Splitting prompts into sentences"):
    # Pre-process the prompt for better segmentation
    processed_prompt = clean_text_for_segmentation(row['prompt'])
    sentences = seg.segment(processed_prompt)
    for sentence in sentences:
        sentence_data.append({
            'type': row['type'],
            'sentence': sentence.strip(), # .strip() to remove leading/trailing whitespace from segmentation
            'sentence_clean': clean_text_basic(sentence).strip(),
            'sentence_token_length': len(clean_text_basic(sentence).split())
        })

jh_sentences_df = pd.DataFrame(sentence_data)

print(f"Original prompts: {len(jh_df)}")
print(f"Total sentences extracted: {len(jh_sentences_df)}")

display(jh_sentences_df.head(10))

In [ ]:
# Filter out sentences below the minimum token length
filtered_jh_sentences_df = jh_sentences_df[(jh_sentences_df['sentence_token_length'] >= MIN_SENTENCE_TOKEN_LENGTH) & (jh_sentences_df['sentence_token_length'] <= MAX_SENTENCE_TOKEN_LENGTH)].copy()
# Filter for only 'jailbreak' sentences
jh_sent_df = filtered_jh_sentences_df[filtered_jh_sentences_df['type'] == 'jailbreak'].copy()

print(f"Total sentences before filtering: {len(jh_sentences_df)}")
print(f"Sentences after filtering (min {MIN_SENTENCE_TOKEN_LENGTH} tokens): {len(filtered_jh_sentences_df)}")
print(f"Jailbreak sentences after filtering: {len(jh_sent_df)}")

display(jh_sent_df.head(10))

In [ ]:
# Stopwords tuned for jailbreak text
JH_STOPWORDS = text.ENGLISH_STOP_WORDS | {
    "please","just","like","would","could","need","want","help",
    "make","give","tell","explain","describe","write","create",
    "example","examples","way","ways","use","using","used",
    "chatgpt", "dan", "generate", "openai"
}

# --- Plot functions ---

def plot_jh_sentence_lengths(ax):
    sns.histplot(
        jh_sent_df["sentence_token_length"],
        bins=40, kde=True, color="steelblue", ax=ax
    )
    ax.set_title("Sentence Length Distribution (Jailbreak)")
    ax.set_xlabel("Token Count")

def plot_jh_unigrams(ax, top_n=20):
    words = " ".join(jh_sent_df["sentence_clean"]).split()
    words = [w for w in words if w not in JH_STOPWORDS and len(w) > 2]

    common = Counter(words).most_common(top_n)
    df = pd.DataFrame(common, columns=["Word", "Count"])

    sns.barplot(x="Count", y="Word", data=df, palette="viridis", hue="Word", ax=ax, legend=False)
    ax.set_title(f"Top {top_n} Unigrams (Jailbreak)")

def plot_jh_bigrams(ax, top_n=20):
    words = " ".join(jh_sent_df["sentence_clean"]).split()
    bigrams = [
        f"{w1} {w2}"
        for w1, w2 in zip(words, words[1:])
        if w1 not in JH_STOPWORDS and w2 not in JH_STOPWORDS
        and len(w1) > 2 and len(w2) > 2
    ]

    common = Counter(bigrams).most_common(top_n)
    df = pd.DataFrame(common, columns=["Bigram", "Count"])

    sns.barplot(x="Count", y="Bigram", data=df, palette="viridis", hue="Bigram", ax=ax, legend=False)
    ax.set_title(f"Top {top_n} Bigrams (Jailbreak)")

def plot_jh_yield_curve(ax, max_MAX_SENTENCE_TOKEN_LENGTH=50):
    thresholds = range(3, max_MAX_SENTENCE_TOKEN_LENGTH+1, 3)
    yields = [
        len(jh_sent_df[jh_sent_df["sentence_token_length"] <= t])
        for t in thresholds
    ]

    ax.plot(thresholds, yields, marker="o", color="steelblue")
    ax.set_title("Usable Sentences vs. Token MAX_SENTENCE_TOKEN_LENGTH")
    ax.set_xlabel("Max Token Length")
    ax.set_ylabel("# Sentences")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

# --- Run grid ---

plot_grid(
    plot_functions=[
        plot_jh_sentence_lengths,
        plot_jh_unigrams,
        plot_jh_bigrams,
        plot_jh_yield_curve
    ],
    n_cols=2,
    figsize_per_plot=(8, 6),
    suptitle="Jailbreak Sentence EDA"
)

In [ ]:
# Generate embeddings for the cleaned sentences
# This might take a little while depending on the number of sentences
print("Generating sentence embeddings...")
jh_embeddings = sentence_model.encode(jh_sent_df['sentence_clean'].tolist(), show_progress_bar=True, batch_size=BATCH_SIZE)

print(f"Embeddings shape: {jh_embeddings.shape}")

In [ ]:
jh_sent_df, kmeans_jh, pca_jh, fig_jh = cluster_and_visualize_sentences(
    dataframe=jh_sent_df,
    sentence_embeddings=jh_embeddings,
    num_clusters=6
)


In [ ]:
cluster_plot_functions = build_cluster_ngram_plots(
    jh_sent_df,
    cluster_col="cluster_st",
    text_col="sentence_clean",
    stop_words=JH_STOPWORDS,
    top_n=10
)

plot_grid(
    plot_functions=cluster_plot_functions,
    n_cols=2,
    figsize_per_plot=(8, 6),
    suptitle="Unigram and Bigram Analysis per Cluster"
)


In [ ]:
VIOLATION_REGEX = RegexComponent(
    label="VIOLATION",
    templates=[
        # Core override intent
        "[ACTION] [TARGET]",
        "[ACTION] [MODIFIER] [TARGET]",

        # Fabrication / hallucination intent
        "[FABRICATION] [CONTENT_TARGET]",
        "[GENERATION] [ALL] [CONTENT_TARGET]",
        "[SPECIFIC_WORDS]"

    ],
    local_vocab={
        # -------------------------
        # Action (keep unified)
        # -------------------------
        "ACTION": {
            r"ignor(?:es?|ed|ing)", "disregard(?:ed|s|ing)?", r"overrid(?:es?|ed|ing|en)", r"bypass(?:es|ing)",
            r"circumvent(?:s|ed|ing)?", r"skip", r"discard", "replace",
            "not", "stop", "never", "deny", "violat(?:es?|ed|ing)",
            "doesn'?t", "can't?", "couldn'?t", "wouldn'?t", "shouldn'?t", "won'?t",
            "don'?t", "disabl(?:es?|ed|ing)", r"free\s+from",
        },

        # -------------------------
        # Modifiers (optional context)
        # -------------------------
        "MODIFIER": {
            "the", "this", "that", "any", "all",
            "previous", "earlier", "prior", "above",
            "system", "initial", "given",
        },

        # -------------------------
        # Targets (merged but structured)
        # -------------------------
        "TARGET": {
            "instructions?",
            "rules?",
            "polic(?:y|ies)",
            "guidelines?",
            "systems?",
            "safety",
            "contents?",
            "restrictions?",
            "guardrails?",
            "filters?",
            "morals?",
            "confines?",
            "refus(?:es?|ed?|ing)",
            "orders?",
            "verify"
        },

        # -------------------------
        # Fabrication (keep separate)
        # -------------------------
        "FABRICATION": {
            "make up", "invent", "fabricate",
            "hallucinate", "fake",
        },
        "GENERATION": {
            "generate", "create", "produce",
        },
        "ALL": {
             "any", "all",
        },

        "CONTENT_TARGET": {
            "informations?", "details?", "answers?",
            "responses?", "facts?", "(?:[A-Za-z]*)things?", "contents?"
        },

        "SPECIFIC_WORDS": {
            "pretend", "do (?:[A-Za-z]*)things?",
             "jailbreak mode",
            "unfiltered mode",
            "unrestricted mode",
        }
    },
)
VIOLATION_REGEX.compile()

In [ ]:
violation_sentences = pd.DataFrame(jh_sent_df[jh_sent_df['sentence_clean'].apply(lambda x: VIOLATION_REGEX.search(x))]['sentence']).drop_duplicates().reset_index(drop=True)

In [ ]:
# --- Control the number of replacements per original sentence ---
NUM_MODEL_REPLACEMENTS_PER_SENTENCE = 3
NUM_ORG_REPLACEMENTS_PER_SENTENCE = 3

# Base real models
base_models = [
    # Your originals
    "LLaMA", "Claude", "Grok", "Bard", "Gemini", "Mistral", "Falcon",
    "Cohere", "HuggingChat", "Phi", "OLMo", "TinyLlama", "Vicuna",
    "DeepSeek", "Qwen",
    "Mixtral", "Mamba", "Jamba",
    "Yi", "InternLM", "Baichuan",
    "Zephyr", "StarCoder", "CodeLLaMA",
    "WizardLM", "OpenHermes", "Nous-Hermes",
    "Pythia", "Dolly", "RedPajama",
    "Gemma", "Reka", "Command-R",
    "MPT", "XGen", "PanGu", "ChatGLM",
    "RWKV", "SmolLM", "MiniCPM",
    "Nemotron", "NeuralChat", "OpenChat",
    "Samba", "DBRX",
]


# Organizations
organization_replacements = [
    # Your originals
    "Google", "Microsoft", "Meta", "Anthropic", "xAI", "Baidu",
    "Tencent", "Amazon", "Alibaba", "Apple", "Nvidia", "DeepMind",

    # Major US / EU labs
    "OpenAI", "Cohere", "Mistral AI", "Hugging Face",
    "Stability AI", "AI21 Labs", "Databricks", "Nomic AI",
    "SambaNova", "Reka AI", "Adept AI", "Inflection AI",
    "EleutherAI", "LAION", "Allen Institute", "AI2",

    # China ecosystem
    "Huawei", "iFlyTek", "SenseTime", "Zhipu AI",
    "ByteDance", "JD.com", "360 AI", "Kunlun Xin",

    # Academia / research groups
    "Stanford CRFM", "Berkeley AI Research", "Berkley",
    "MIT CSAIL", "Carnegie Mellon LTI", "Tsinghua KEG Lab",

    # Hardware / infra players
    "AMD", "Intel", "Qualcomm", "Cerebras", "Graphcore",

    # Cloud providers
    "Oracle", "IBM", "Snowflake", "Salesforce", "OpenRouter",

    # Open-source communities
    "Red Hat", "Apache", "Linux",
]

# -----------------------------
# Fictional model generation
# -----------------------------
suffixes = ["GPT", "AI", "Chat", "Bot", "LLM", "ML", "LM"]

# Single-letter models
single_letter_models = [f"{letter}{random.choice(suffixes)}" for letter in "abcdefghijklmnopqrstuvwxyz"]

# Dictionary-word models
english_words = [w.capitalize() for w in words.words() if w.isalpha() and 3 <= len(w) <= 8]
dict_word_models = [f"{random.choice(english_words)}{random.choice(suffixes)}" for _ in range(30)]

# Combine all models
model_replacements = base_models + single_letter_models + dict_word_models

# -----------------------------
# Keyword rules
# -----------------------------
model_keywords = {
    "DAN": {"match_type": "full"},
    "chat": {"match_type": "prefix"},
    "gpt": {"match_type": "suffix"},
    "bot": {"match_type": "suffix"},
    "ai": {"match_type": "suffix"},
    "llm": {"match_type": "suffix"},
    "ml": {"match_type": "suffix"},
}

organization_keywords = {
    key: {"match_type": "substring"}
    for key in organization_replacements
}


# -----------------------------
# Version generator (returns raw version string)
# -----------------------------
def generate_version():
    digit = str(random.randint(1, 9))
    subdigit = str(random.randint(0, 9))
    alpha = random.choice(string.ascii_lowercase)

    version_formats = [
        "", # No version
        "digit",
        "digit.digit",
        "vdigit",
        "vdigit.digit",
        "digit_alpha",
        "alpha_digit"
    ]

    pattern = random.choice(version_formats)

    version_str = ""
    if pattern == "digit":
        version_str = digit
    elif pattern == "digit.digit":
        version_str = f"{digit}.{subdigit}"
    elif pattern == "vdigit":
        version_str = f"v{digit}"
    elif pattern == "vdigit.digit":
        version_str = f"v{digit}.{subdigit}"
    elif pattern == "digit_alpha":
        version_str = f"{digit}{alpha}"
    elif pattern == "alpha_digit":
        version_str = f"{alpha}{digit}"

    return version_str

# -----------------------------
# Replacement function
# -----------------------------
def replace_word(word, replacement, keywords):
    lw = word.lower()
    for kw, rule in keywords.items():
        mt = rule["match_type"]
        kwl = kw.lower()
        if (mt == "full" and lw == kwl) or \
           (mt == "prefix" and lw.startswith(kwl)) or \
           (mt == "suffix" and lw.endswith(kwl)) or \
           (mt == "substring" and kwl in lw):
            return replacement
    return word

# -----------------------------
# Expansion loop
# -----------------------------
expanded_sentences_data = []

for sentence_text in tqdm(violation_sentences['sentence'], desc="Expanding sentences"):
    words_in_sentence = sentence_text.split()

    sampled_models = random.sample(model_replacements, min(len(model_replacements), NUM_MODEL_REPLACEMENTS_PER_SENTENCE))
    sampled_orgs = random.sample(organization_replacements, min(len(organization_replacements), NUM_ORG_REPLACEMENTS_PER_SENTENCE))

    for model_name_candidate in sampled_models:
        raw_version = generate_version()
        version_prefix = ""
        if raw_version:
            if raw_version[0].isalpha() or raw_version[-1].isalpha():
                version_prefix = random.choice(["-", " "])
            elif raw_version[0].isdigit():
                version_prefix = ""
        versioned_model = f"{model_name_candidate}{version_prefix}{raw_version}"

        temp_model_replaced_words = []
        model_was_introduced_by_replacement = False # Flag to track if the model name was placed via keyword replacement

        for i, word in enumerate(words_in_sentence):
            next_word = words_in_sentence[i+1] if i+1 < len(words_in_sentence) else None
            potential_replacement = replace_word(word, versioned_model, model_keywords)

            if (potential_replacement != word) and (next_word and next_word.lower() == 'mode'):
                temp_model_replaced_words.append(word) # Don't replace the model keyword if followed by 'Mode'
            else:
                if potential_replacement != word:
                    model_was_introduced_by_replacement = True # Model name was put into the sentence
                temp_model_replaced_words.append(potential_replacement)

        for org_name_candidate in sampled_orgs:
            final_words_after_all_replacements = []
            org_was_introduced_by_replacement = False # Flag to track if the org name was placed via keyword replacement

            for w in temp_model_replaced_words:
                potential_replacement = replace_word(w, org_name_candidate, organization_keywords)
                if potential_replacement != w:
                    org_was_introduced_by_replacement = True # Org name was put into the sentence
                final_words_after_all_replacements.append(potential_replacement)

            # The final sentence text is simply the joined words after all replacements
            final_sentence_text = " ".join(final_words_after_all_replacements)

            # Populate metadata conditionally
            model_meta = versioned_model if model_was_introduced_by_replacement else None
            org_meta = org_name_candidate if org_was_introduced_by_replacement else None

            expanded_sentences_data.append({
                "sentence": final_sentence_text,
                "model_generated_with": model_meta,
                "org_generated_with": org_meta
            })

# Deduplicate, now considering the new metadata columns for uniqueness
expanded_violation_df = pd.DataFrame(expanded_sentences_data).drop_duplicates(subset=['sentence', 'model_generated_with', 'org_generated_with'])

print(f"Original unique violation sentences: {len(violation_sentences.drop_duplicates())}")
print(f"Expanded unique violation sentences (with metadata): {len(expanded_violation_df)}")
display(expanded_violation_df)


# Merge Jailbreak/Salad

In [ ]:
# Prepare the augmented jailbreak sentences
jh_augmented_unified_df = expanded_violation_df.copy()
jh_augmented_unified_df['text'] = jh_augmented_unified_df['sentence']
jh_augmented_unified_df['category'] = PromptEntity.VIOLATION.value
jh_augmented_unified_df['dataset_source'] = 'jackhhao_jailbreak'
jh_augmented_unified_df['source'] = "augmented"

def create_metadata_string(row):
    model_meta = row['model_generated_with']
    org_meta = row['org_generated_with']

    if model_meta and org_meta:
        return f"Model: {model_meta}, Org: {org_meta}"
    elif model_meta:
        return f"Model: {model_meta}"
    elif org_meta:
        return f"Org: {org_meta}"
    else:
        return ""

jh_augmented_unified_df['metadata'] = jh_augmented_unified_df.apply(create_metadata_string, axis=1)
jh_augmented_unified_df = jh_augmented_unified_df[['text', 'category', 'dataset_source', 'source', 'metadata']]

# Prepare original jailbreak sentences
jh_unified_df = violation_sentences.copy()
jh_unified_df['category'] = PromptEntity.VIOLATION.value
jh_unified_df = jh_unified_df.rename(columns={'sentence': 'text'})
jh_unified_df['dataset_source'] = 'jackhhao_jailbreak'
jh_unified_df['source'] = "jackhhao_jailbreak"
jh_unified_df['metadata'] = ""
jh_unified_df = jh_unified_df[['text', 'category', 'dataset_source', 'source', 'metadata']]


# Prepare Salad-Data sentences, retaining original 'source' column
sd_unified_df = sd_sent_df.copy()
sd_unified_df = sd_unified_df.rename(columns={'question': 'text', 'entity': 'category', '3-category': 'metadata'})
sd_unified_df['dataset_source'] = 'salad_data'
sd_unified_df = sd_unified_df[['text', 'category', 'dataset_source', 'source', 'metadata']]

# Concatenate all three dataframes
jb_df = pd.concat([jh_unified_df, jh_augmented_unified_df, sd_unified_df], ignore_index=True)

# Drop duplicates on text
jb_df = jb_df.drop_duplicates(subset=['text'])

print(f"Original Jailbreak sentences: {len(jh_unified_df)}")
print(f"Augmented Jailbreak sentences: {len(jh_augmented_unified_df)}")
print(f"Original Salad-Data sentences: {len(sd_unified_df)}")
print(f"Total Unified DataFrame size: {len(jb_df)}")

df_sampled = (
    jb_df.groupby("source", group_keys=False)
      .sample(n=10, replace=False)
)
display(df_sampled)


# Benign Prompts

In [ ]:
splits = {'test': 'all/test-00000-of-00001.parquet', 'validation': 'all/validation-00000-of-00001.parquet', 'dev': 'all/dev-00000-of-00001.parquet', 'auxiliary_train': 'all/auxiliary_train-00000-of-00001.parquet'}
cais_df = pd.read_parquet("hf://datasets/cais/mmlu/" + splits["test"])

In [ ]:
cais_df.head()

In [ ]:
safe_subjects = [
    "abstract_algebra",
    "astronomy",
    "college_mathematics",
    "elementary_mathematics",
    "formal_logic",
    "high_school_mathematics",
    "high_school_statistics",
    "econometrics",
    "high_school_macroeconomics",
    "philosophy",
    "world_religions",
    "prehistory",
    "world_history",
    "high_school_geography",
    "high_school_computer_science",
    "global_facts",
    "logical_fallacies",
    "nutrition",
    "high_school_physics",
    "machine_learning",
]

cais_safe_df = cais_df[cais_df['subject'].isin(safe_subjects)]
cais_safe_df = cais_safe_df.rename(columns={'question': 'text', 'subject': 'metadata'})

cais_safe_df = cais_safe_df[['text', 'metadata']]
cais_safe_df['dataset_source'] = 'cais_mmlu'
cais_safe_df['source'] = 'cais_mmlu'
cais_safe_df['category'] = PromptEntity.BENIGN.value
cais_safe_df

In [ ]:
print("Generating embeddings for cais_safe_df...")
cais_embeddings = sentence_model.encode(cais_safe_df['text'].tolist(), show_progress_bar=True, batch_size=BATCH_SIZE)

print(f"cais_safe_df embeddings shape: {cais_embeddings.shape}")

In [ ]:
cais_sent_df, kmeans_cais, pca_cais, fig_cais = cluster_and_visualize_sentences(
    dataframe=cais_safe_df,
    sentence_embeddings=cais_embeddings,
    num_clusters=10,
    target_label="text"
)

In [ ]:
hm_df = pd.read_json("hf://datasets/teknium/OpenHermes-2.5/openhermes2_5.json")

In [ ]:
hm_df.head()

In [ ]:
def process_row(row):
    sentence_data = []

    human_prompt = None
    if 'conversations' in row and isinstance(row['conversations'], list):
        for turn in row['conversations']:
            if isinstance(turn, dict) and turn.get('from') == 'human':
                human_prompt = turn.get('value')
                break

    if not human_prompt:
        return sentence_data

    # 1. Clean the full prompt (removed segmentation)
    processed_prompt = clean_text_for_segmentation(human_prompt)

    # 2. Get metadata
    original_source = row.get('source')
    dataset_source = 'openhermes'

    sentence_data.append({
        'text': processed_prompt.strip(),
        'source': original_source,
        'dataset_source': dataset_source,
        'category': PromptEntity.BENIGN.value,
        'metadata': None
    })

    return sentence_data

# Sample hm_df to reduce processing time, if desired
sampled_hm_df = hm_df.sample(n=200000, random_state=42)
rows = [row for _, row in sampled_hm_df.iterrows()]

sentence_data = []

with ProcessPoolExecutor() as executor:
    for result in tqdm(executor.map(process_row, rows, chunksize=500), total=len(rows), desc="Extracting human prompts"):
        if result:
            sentence_data.extend(result)

hm_user_df = pd.DataFrame(sentence_data)
print(f"Total human prompts extracted: {len(hm_user_df)}")
display(hm_user_df.head())

In [ ]:
# Encode hm
hm_embeddings = sentence_model.encode(hm_user_df['text'].tolist(), show_progress_bar=True, batch_size=BATCH_SIZE)
print(f"OpenHermes embeddings shape: {hm_embeddings.shape}")

In [ ]:
# We are creating large clusters
hm_sent_df, kmeans_hm, pca_hm, fig_hm = cluster_and_visualize_sentences(
    dataframe=hm_user_df,
    sentence_embeddings=hm_embeddings,
    num_clusters=30,
    target_label="text"
)

In [ ]:
# --- Create nshot_df for multi-shot detection --- START

# 1. Prepare Cais MMLU data
# Re-derive Cais MMLU data to retain 'choices' and 'answer' for formatting
cais_mmlu_raw = cais_df[cais_df['subject'].isin(safe_subjects)].copy()
cais_mmlu_raw = cais_mmlu_raw.rename(columns={'question': 'original_question_text', 'subject': 'metadata'})
cais_nshot_entries = []
for idx, row in tqdm(cais_mmlu_raw.iterrows(), total=len(cais_mmlu_raw), desc="Formatting Cais MMLU for n-shot"):
    question_text = str(row['original_question_text'])
    choices = row['choices'].tolist()
    answer_idx = row['answer']

    # Ensure answer_idx is valid and choices is a list
    answer_text = "[Answer not available]"
    if isinstance(choices, list) and 0 <= answer_idx < len(choices):
        answer_text = choices[answer_idx]

    # Format: [HUMAN]:\n(question)\n[ASSISTANT]:\n(answer)
    formatted_text = f"[HUMAN]:\n{question_text}\n[ASSISTANT]:\n{answer_text}"
    clean_formatted_text = clean_text_basic(formatted_text)
    token_len = len(clean_formatted_text.split())

    # Filter only by MAX_SENTENCE_TOKEN_LENGTH
    if token_len <= MAX_SENTENCE_TOKEN_LENGTH:
        cais_nshot_entries.append({
            'text': formatted_text,
            'text_clean': clean_formatted_text,
            'token_length': token_len,
            'source': 'cais_mmlu',
            'dataset_source': 'cais_mmlu',
            'category': PromptEntity.NSHOT.value,
            'metadata': row['metadata'] # Keep original subject as metadata
        })

cais_nshot_df = pd.DataFrame(cais_nshot_entries)

# Sample 2000 examples if available
if len(cais_nshot_df) > 2000:
    cais_nshot_df = cais_nshot_df.sample(n=2000, random_state=42)


# 2. Prepare OpenHermes data (human-GPT conversation pairs)
def process_openhermes_for_nshot_pairs(row):
    nshot_pairs = []
    conversations = row.get('conversations', [])
    if not isinstance(conversations, list):
        return nshot_pairs

    for i in range(len(conversations) - 1):
        turn1 = conversations[i]
        turn2 = conversations[i+1]

        if turn1.get('from') == 'human' and turn2.get('from') == 'gpt':
            human_text = turn1.get('value', '').strip()
            gpt_text = turn2.get('value', '').strip()

            if human_text and gpt_text:
                # Format: [HUMAN]:\n(user/question)\n[ASSISTANT]:\n(answer/gpt)
                combined_text = f"[HUMAN]:\n{human_text}\n[ASSISTANT]:\n{gpt_text}"
                clean_combined_text = clean_text_basic(combined_text)
                token_len = len(clean_combined_text.split())

                if token_len <= MAX_SENTENCE_TOKEN_LENGTH:
                    nshot_pairs.append({
                        'text': combined_text,
                        'text_clean': clean_combined_text,
                        'token_length': token_len,
                        'source': row.get('source'),
                        'dataset_source': 'openhermes',
                        'category': PromptEntity.NSHOT.value,
                        'metadata': f"turn_type:human-gpt_pair"
                    })
    return nshot_pairs

# Using sampled_hm_df (200k rows) for efficiency
hm_nshot_pairs_data = []
for idx, row in tqdm(sampled_hm_df.iterrows(), total=len(sampled_hm_df), desc="Processing OpenHermes for n-shot pairs"):
    hm_nshot_pairs_data.extend(process_openhermes_for_nshot_pairs(row))

hm_nshot_df = pd.DataFrame(hm_nshot_pairs_data)

# Remove potential duplicates before sampling
hm_nshot_df = hm_nshot_df.drop_duplicates(subset=['text'])

# Sample 2000 examples if available
if len(hm_nshot_df) > 2000:
    hm_nshot_df = hm_nshot_df.sample(n=2000, random_state=42)


# 3. Concatenate and deduplicate final nshot_df
nshot_df = pd.concat([cais_nshot_df, hm_nshot_df], ignore_index=True)
nshot_df = nshot_df.drop_duplicates(subset=['text'])
# Drop unneeded debug columns
nshot_df = nshot_df.drop(columns=['text_clean', 'token_length'])

print(f"\nTotal Cais MMLU examples for n-shot (after filtering and sampling): {len(cais_nshot_df)}")
print(f"Total OpenHermes examples for n-shot (after filtering and sampling): {len(hm_nshot_df)}")
print(f"Total nshot_df (deduplicated): {len(nshot_df)}")

# Group by dataset_source and display it
print("\n--- nshot_df grouped by dataset_source ---\n")
display(nshot_df.groupby("dataset_source").sample(n=3, replace=True))

In [ ]:
fp_df = pd.read_parquet("hf://datasets/argilla/FinePersonas-v0.1/data/train-00000-of-00012.parquet")
fp_df.head()

In [ ]:
fp_processed_df = fp_df.drop(columns=['id']).rename(columns={'persona': 'text'})[['text', 'labels']]

# Extract unique labels by exploding the list of labels and then getting unique values
all_labels = fp_processed_df['labels'].progress_apply(lambda x: eval(x) if isinstance(x, str) else x).explode()
unique_labels = all_labels.unique()

print(f"Total rows in FinePersonas dataset: {len(fp_processed_df)}")
print(f"Number of unique labels: {len(unique_labels)}")
print("First 5 unique labels:")
display(fp_processed_df.head())

In [ ]:
def get_categorizer():
    # Priority Order: Most specialized/dangerous personas first.
    # Each category here maps logically to one or more PromptEntity values.
    CATEGORIES_RAW = [

      ('Cyber & IT', [
          # Offensive & Defensive Security
          'Cyber(?:[A-Za-z]*)', 'Malware', 'Penetration Tester',
          'Hacker', 'Security Researcher', 'Digital', 'Incident Response',
          'Crypto(?:[A-Za-z]*)', 'Information Security', 'InfoSec',

          # Low-Level & Systems Programming
          'Software', 'Programmer',
          'Systems Architect', 'Reverse Engineering', 'Assembly Language', 'C Programming',
          'C++', 'Rust Programming', 'Kernel Developer', 'Firmware', 'Embedded Systems',
          'Programming',

          # Infrastructure & Networking
          'Network Administrator', 'Systems Administrator',
          'SysAdmin', 'Telecommunications', 'Cloud Architect', 'DevOps',
          'Site Reliability Engineering', 'SRE', 'Database',
          'DBA', 'Server Management',

          # Specialized Tech
          'Web Developer', 'Full Stack', 'Backend Developer', 'API Developer',
          'Automation Specialist', 'VBA Developer', 'Scripting', 'PowerShell',
          'Technologist', 'Electronics', 'Hardware Engineer', 'Data Analyist',
          'Blockchain', 'Smart Contracts', 'Artificial Intelligence', 'Machine Learning',
          'IT Specialist', 'IT Professional', 'Tech Professional', 'Technical',
          'Technical Specialist', 'Computing', "Computer", 'Linux',
      ]),


      ('CBRN & High-Risk Science', [
          # Nuclear & Radiological
          'Nuclear', 'Radiolog(?:[A-Za-z]*)', '(?:[A-Za-z]*)Physic(?:[A-Za-z]*)',

          # Biological & Infectious
          'Infectious', 'Disease', 'Virol(?:[A-Za-z]*)', 'Bacteriol(?:[A-Za-z]*)',
          '(?:[A-Za-z]*)Biolog(?:[A-Za-z]*)', 'Biomedical',

          # Chemical & Materials
          'Chemistry', 'Chemical', 'Chemist', 'Toxicol(?:[A-Za-z]*)',
          'Materials Science', 'Nanotechnology', 'Pharmacology',

          # Aerospace & Dual-Use Tech
          'Aerospace', 'Aviation', 'Space Exploration', 'Propulsion',
          'Scientific', 'Scientist'
      ]),


      ('Healthcare & Medical', [
          'Medical Professional', 'Healthcare Specialist', 'Medicine', 'Medical Expert',
          'Medical Practitioner', 'Doctor', 'Surgeon', 'Allergy Specialist', 'Ophthalmology',
          'Gastroenterology', 'Nephrology Specialist', 'Dermatology', 'Dentistry',
          'Mental Health Professional', 'Psychology', 'Psychiatrist', 'Sleep', 'Anatomy',
          'Audiology Related', 'Speech-Language Specialist', 'Healthcare', 'Medical',
          'Neurologist', 'Psychologist',

          # Additions
          'Eating Disorder Specialist', 'OCD Specialist', 'Mental Health Expert',
          'Neuroscience', 'Cognitive Science',

          'Mental Health', 'ADHD', 'Cognitive Science', 'Behavioral Science',

          'Veterinary Medicine', 'Animal Health', 'Animal Care Specialist',
          'Animal Behavior', 'Equine Expert', 'Equestrian Professional',
          'Fish Care Professional', 'Aquarium Enthusiast'
      ]),

      # -------------------------------------------------
      # 4. Legal, Finance & Policy
      # -------------------------------------------------
      ('Legal & Policy', [
        # Core legal roles
        '(?:[A-Za-z]*)Legal',
        'Lawyer',
        'Attorney',
        # Legal specialties
        'Law',

        # Government & policy
        'Policy Analyst',
        'Public Policy',
        'Regulatory Affairs',
        'Compliance Officer',
        'Government Affairs',
        'Public Administration',
        'Urban Planning',
        'Transportation Development',

        # Justice system
        'Magistrate',
        'Mediator',
        'Arbitrator',
        'Court Clerk',
        'Legal Investigator',

        # HR & compliance
        'Human Resources',
        'HR Specialist',
        'Labor Relations',

        # Additional policy-adjacent
        'Legislative Analyst',
        'Political Science',
        'Public Sector Professional',
        'Civic Administration'
    ]),

      ('Business, Finance & Management', [
          # Core business & management
          'Business',
          'Management',
          'Manager',
          'Executive',
          'Leadership',
          'Professional Development',

          # Finance & accounting
          'Financ(?:[A-Za-z]*)',
          'Economics',
          'Accountant',
          'Accounting',
          'Auditor',
          'Investment',
          'Banking',
          'Wealth',
          'Tax Specialist',

          # Marketing & sales
          'Marketing',
          'Brand Strategist',
          'Market Research(?:[A-Za-z]*)',
          'Sales',
          'Entrepreneur',
          'Startup',

          # Corporate operations
          'Project Manager',
          'Operations',
          'Logistics',
          'Supply Chain',
          'Procurement',
          'Strategic Planning',

          # HR (business side)
          'Talent Management',
          'Recruiter',
          'Organizational Development',

          # Additional business-adjacent
          'Consultant',
          'Business Coach',
          'Corporate Trainer',
          'Customer Success',
          'Client Relations'
      ]),




      ('Skilled Trades & Industrial Work', [
          'Construction', 'HVAC', 'Manufacturing',
          'Woodworking','Craftsmanship',
      ]),


      ('Humanities & Social Sciences', [
          'Histor(?:[A-Za-z]*)', 'Philosoph(?:[A-Za-z]*)', 'Ethics',
          'Religion', 'Creationism', 'Anti-Evolutionism',
          'Anthro(?:[A-Za-z]*)', 'Archaeol(?:[A-Za-z]*)', 'Linguistics',
          'Sociolog(?:[A-Za-z]*)', 'Social Science',
          '(?:[A-Za-z]*)Cultur(?:[A-Za-z]*)', 'Human',

          # Additions
          'Social Sciences', 'Language',
          'Disability', 'Social Justice', 'Accessibility',
          'Diversity and Inclusion', 'Social Equity',

          'Food', 'Culinary',
      ]),


      ('Environment & Wildlife', [
          'Ecol(?:[A-Za-z]*)', 'Herpetol(?:[A-Za-z]*)', 'Arachno(?:[A-Za-z]*)', 'Paleont(?:[A-Za-z]*)',
          'Geol(?:[A-Za-z]*)', 'Earth', 'Marine',
          'Climate Change', 'Ornitho(?:[A-Za-z]*)', 'Astron(?:[A-Za-z]*)', 'Agricul(?:[A-Za-z]*)', 'Viticul(?:[A-Za-z]*)',
          'Sustainabil(?:[A-Za-z]*)', 'Renewable',

          # Additions
          'Wildlife', 'Conservation', 'Environment(?:[A-Za-z]*)',
          'Climate',
          'Geography', 'Cartography', 'Geomorphology',
      ]),


      ('Arts, Design & Media', [
          'Graphic', 'Visual Arts', 'Photography', 'Music', 'Fashion',
          'Textile', 'Architecture', 'Dance',
          'Art Historian', 'Content Curator', 'Journalism',

          # Additions
          'Art', 'Crafts', 'Audio', 'Editor', 'Design'
      ]),

      ('Mathematics', [
          'Math', "Math(?:[A-Za-z]*)", "Algebra", "Geometry", "Calculus", "Trigonometry",
          "Statist(?:[A-Za-z]*)",
      ]),

      ('Education & Academia', [
          'Educator', 'Teacher', 'Instructor', 'Teaching', 'Professor',
          'Montessori', 'Curriculum', 'Academic',
          'Research', 'Scientist', 'Literacy', 'Homeschooling', 'Test Preparation',
          'Education',

          # Additions
          'Research', 'Researcher', 'Student'
      ]),





      ]


    compiled_hierarchy = []
    for category_name, keywords in CATEGORIES_RAW:
        # \b ensures we match "Law" but not "Lawn"
        pattern_string = build_alternation(keywords)
        pattern = re.compile(rf"\b{pattern_string}s?\b", re.IGNORECASE)
        compiled_hierarchy.append((category_name, pattern))

    return compiled_hierarchy

HIERARCHY = get_categorizer()

def categorize_persona(row) -> Optional[str]:
    labels = row.get('labels_list', [])
    text = row.get('text', "")
    labels_str = " ".join([str(l) for l in labels]) if labels else ""

    # LEVEL 1: High-confidence label matches
    for category_name, pattern in HIERARCHY:
        if pattern.search(labels_str):
            return category_name

    # LEVEL 2: Descriptive text matches
    if text:
        for category_name, pattern in HIERARCHY:
            if pattern.search(text):
                return category_name

    # FALLBACK
    if labels or text:
       return PromptEntity.PERSONA.value

    return None

# 1. Apply categorization
fp_processed_df['metadata'] = fp_processed_df.progress_apply(categorize_persona, axis=1)

# 2. Filter and Unify
fp_unified_df = (
    fp_processed_df.dropna(subset=['metadata'])
    [['text', 'metadata']]
    .copy()
)

fp_unified_df['dataset_source'] = 'finepersonas'
fp_unified_df['source'] = 'finepersonas'
fp_unified_df['category'] = PromptEntity.PERSONA.value

In [ ]:
# 3. Final Check
print(f"Total categorized entries: {len(fp_unified_df)}")
print(fp_unified_df['metadata'].value_counts())
display(fp_unified_df.head())

In [ ]:
# Encode fp
fp_sample_df = fp_unified_df.sample(n=20000)
fp_embeddings = sentence_model.encode(fp_sample_df['text'].tolist(), show_progress_bar=True, batch_size=BATCH_SIZE)
print(f"FinePersonas embeddings shape: {fp_embeddings.shape}")

In [ ]:
fp_sent_df, kmeans_fp, pca_fp, fig_fp = cluster_and_visualize_sentences(
    dataframe=fp_sample_df,
    sentence_embeddings=fp_embeddings,
    num_clusters=10,
    target_label="text"
)

# Label OpenHermes

In [ ]:
"""
array([nan, 'glaive-code-assist', 'UnnaturalInstructions',
       'EvolInstruct_70k', 'airoboros2.2', 'metamath', 'CamelAI',
       'cot_alpaca_gpt4', 'platypus', 'GPT-4 Comparison Data',
       'CogStackMed', 'lmsys1m', 'Econ_domain_expert',
       'LMSys Chatbot Arena', 'caseus_custom'], dtype=object)
"""

# code-assist -> Cyber & IT metadata
HM_TO_FPCAT = {
    'glaive-code-assist': 'Cyber & IT',
    'CogStackMed': 'Healthcare & Medical',
    'Econ_domain_expert': 'Business, Finance & Management',
    'metamath': 'Mathematics',
}


hm_sent_df['metadata'] = hm_sent_df['source'].map(HM_TO_FPCAT)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import MiniBatchKMeans

def auto_label_sampled_clusters(
    target_df,
    target_embeddings,
    source_dfs,
    batch_size: int = 5000,       # rows processed per iteration
    n_clusters: int = 10,          # clusters per batch
    top_n_similar: int = 5,
    similarity_threshold: float = 0.6,
    random_state: int = 42,
):
    target_df = target_df.copy()
    rng = np.random.default_rng(random_state)

    # ── 1. Identify unlabeled rows ────────────────────────────────────────────
    # Ensure 'metadata' column exists for target_df, create if not to avoid KeyError
    if 'metadata' not in target_df.columns:
        target_df['metadata'] = None

    needs_label_mask = target_df['metadata'].isna().values
    needs_label_idx  = np.where(needs_label_mask)[0]

    if len(needs_label_idx) == 0:
        print("Nothing to label.")
        return target_df

    print(f"{len(needs_label_idx):,} unlabeled rows → "
          f"batches of {batch_size}, {n_clusters} clusters each")

    # Shuffle so batches aren't positionally biased
    rng.shuffle(needs_label_idx)

    label_array = target_df['metadata'].to_numpy(dtype=object)

    # ── 2. Iterate over batches ───────────────────────────────────────────────
    batches = [
        needs_label_idx[i : i + batch_size]
        for i in range(0, len(needs_label_idx), batch_size)
    ]

    for batch_num, batch_idx in enumerate(tqdm(batches, desc="Batches")):
        batch_embeddings = target_embeddings[batch_idx]  # (≤batch_size, dim)

        k = min(n_clusters, len(batch_idx))  # can't have more clusters than rows

        # ── 3. Cluster this batch ─────────────────────────────────────────────
        kmeans = MiniBatchKMeans(
            n_clusters=k,
            batch_size=min(1024, len(batch_idx)),
            random_state=random_state,
            n_init=3,
        )
        cluster_ids = kmeans.fit_predict(batch_embeddings)  # (batch_size,)
        centroids   = kmeans.cluster_centers_               # (k, dim)

        # ── 4. Label each centroid ────────────────────────────────────────────
        centroid_labels = [None] * k

        for c_idx in range(k):
            centroid_vec  = centroids[c_idx].reshape(1, -1)
            candidate_labels = []

            for source_df, source_embeddings, label_col in source_dfs:
                # Check if the label_col exists in the current source_df
                if label_col not in source_df.columns:
                    continue # Skip this source if it doesn't have the required label_col

                sims        = cosine_similarity(centroid_vec, source_embeddings)[0]
                top_indices = sims.argsort()[-top_n_similar:][::-1]

                for top_idx in top_indices:
                    if sims[top_idx] >= similarity_threshold:
                        label = source_df.iloc[top_idx][label_col]
                        if label is not None:
                            candidate_labels.append(str(label))

            if candidate_labels:
                centroid_labels[c_idx] = list(set(candidate_labels))[0]

        # ── 5. Broadcast centroid label → every row in that cluster ───────────
        for c_idx, label in enumerate(centroid_labels):
            rows_in_cluster = batch_idx[cluster_ids == c_idx]
            label_array[rows_in_cluster] = label

    target_df['metadata'] = label_array
    return target_df


# ── Usage ─────────────────────────────────────────────────────────────────────
sources_for_labeling = [
    (fp_unified_df, fp_embeddings,  'metadata'),
    (cais_safe_df,  cais_embeddings, 'metadata'),
    (sd_unified_df,    sd_embeddings,   'metadata'), # Changed sd_sent_df to sd_unified_df
]

hm_user_df_labeled = auto_label_sampled_clusters(
    target_df=hm_user_df,
    target_embeddings=hm_embeddings,
    source_dfs=sources_for_labeling,
    batch_size=5000,
    n_clusters=20,
    similarity_threshold=0.6,
)

# Full

In [ ]:
full_df = pd.concat([jb_df, cais_safe_df, hm_user_df_labeled, fp_sample_df, nshot_df], ignore_index=True)
full_df = full_df.drop_duplicates(subset=['text'])

print(f"Total Unified DataFrame size after adding prompts: {len(full_df)}")

display(full_df.groupby("source").sample(n=3, replace=True))

In [ ]:
full_df.to_parquet("prompt_nlp.parquet")

# Cleanup 2


In [ ]:
full_df = pd.read_parquet("hf://datasets/DerivedFunction/temporary-prompt-nlp/prompt_nlp.parquet")

In [ ]:
WRAP_KEYS = ['text', 'category', 'dataset_source', 'source', 'metadata']

In [ ]:
def create_n_shot_prompts(nshot_df, max_shots=4, related_prob=0.8):
    """
    Groups individual HUMAN/ASSISTANT pairs into multi-shot sequences.
    Standardizes output using WRAP_KEYS for the master pipeline.
    """
    n_shot_records = []

    # 1. Group by metadata to facilitate "Related" sampling
    meta_groups = {m: grp.to_dict('records') for m, grp in nshot_df.groupby('metadata')}
    all_records = nshot_df.to_dict('records')

    # 2. Prepare indices for iteration
    available_indices = list(range(len(all_records)))
    np.random.shuffle(available_indices)

    # Use tqdm to track progress as we consume indices
    pbar = tqdm(total=len(available_indices), desc="Clustering N-Shots")

    while available_indices:
        num_shots = np.random.randint(1, max_shots + 1)
        is_related = np.random.random() < related_prob

        group = []
        if is_related and available_indices:
            # Seed the group with a random record
            idx = available_indices.pop()
            seed_record = all_records[idx]
            group.append(seed_record)
            pbar.update(1)

            # Try to fill with metadata siblings
            meta = seed_record['metadata']
            siblings = [r for r in meta_groups[meta] if r != seed_record]

            # Filter siblings to only those still 'available' if you want
            # absolute deduplication, but usually sampling from the group is fine.
            num_to_add = min(num_shots - 1, len(siblings))
            if num_to_add > 0:
                selected_siblings = np.random.choice(siblings, num_to_add, replace=False)
                group.extend(selected_siblings)
                # Note: These aren't popped from available_indices to maximize
                # metadata cohesion, but you can pop them if you want 1:1 data usage.
        else:
            # PURE RANDOMNESS
            for _ in range(num_shots):
                if not available_indices: break
                group.append(all_records[available_indices.pop()])
                pbar.update(1)

        if not group: break

        # 3. Final Assembly using WRAP_KEYS logic
        # 'text' is already a list of strings here.
        # Other keys are aggregated into lists to match the 'one row per cluster' schema.
        n_shot_records.append({
            "text": [item['text'] for item in group],
            "category": [PromptEntity.NSHOT.value] * len(group),
            **{k: [item[k] for item in group] for k in WRAP_KEYS if k != 'text' and k != 'category'}
        })

    pbar.close()

    # Shuffle and return
    return pd.DataFrame(n_shot_records).sample(frac=1).reset_index(drop=True)

# Generate the clustered DF
nshot_df = create_n_shot_prompts(
    full_df[full_df['category'] == PromptEntity.NSHOT.value],
    related_prob=0.75  # 25% chance of a "random mix" sequence
)
nshot_df.sample(10)

In [ ]:
# Pre-compile patterns
SPECIFIC_KEYWORDS_PATTERN = re.compile(
    r"(?:" + "|".join([
        r",\s+likely", r",\s+possibly", r",\s+particularly",
        r",\s+focused", r",\s+specializing", r"\s+who\s+",
        r"\s+with\s+a\s+focus", r"\s+examining\s+",
        r",\s+interested", r"\s+focusing\s+"
    ]) + r")",
    flags=re.IGNORECASE
)
COMMA_SPLIT_PATTERN = re.compile(r",\s+")

def clean_persona_span(text, max_tokens=15):
    if not text: return []

    # 1. Surgical split
    # Using 'maxsplit=1' inside split for a minor speed boost
    parts = SPECIFIC_KEYWORDS_PATTERN.split(text, maxsplit=1)
    clean_text = parts[0].strip().rstrip(',')

    # 2. Token-gated comma split
    # Only calculate split() if a comma actually exists to save cycles
    if "," in clean_text:
        tokens = clean_text.split()
        if len(tokens) > max_tokens:
            clean_text = COMMA_SPLIT_PATTERN.split(clean_text, maxsplit=1)[0].strip()

    # 3. Period attachment
    if len(clean_text) < len(text):
        clean_text = clean_text.rstrip('.') + "."

    return [clean_text]

def create_cleaned_persona_df(df, token_threshold=15):
    # Filter first to reduce the iteration set
    raw_records = df[df['category'] == PromptEntity.PERSONA.value].to_dict('records')

    processed_data = []

    # Iterating over a list of dicts is 10x-50x faster than iterrows()
    for row in tqdm(raw_records, desc="Processing Personas"):
        cleaned_array = clean_persona_span(row['text'], max_tokens=token_threshold)

        # We reuse the existing row dict to avoid creating new objects
        row['text'] = cleaned_array
        # use wrap_keys

        processed_data.append({
            "text": cleaned_array,
            **{k: [row[k]] for k in WRAP_KEYS if k != "text"}
        })

    return pd.DataFrame(processed_data).drop_duplicates(subset=['text'])

# Execution
persona_df = create_cleaned_persona_df(full_df)
persona_df.sample(10)

In [ ]:
ENGLISH_WORDS = set(w.lower() for w in nltk.corpus.words.words())

# 2. Optimized Global Patterns
ALL_CAPS_PATTERN = re.compile(r'\b[A-Z]{2,}\b')

# SURGICAL STRIP: Remove leading/trailing symbols like ===, ###, !!!, >>>
# But explicitly PROTECT: . , : ; " ' ( )
# ^[^\w\.\,\:\;\"\'\(\)]+ matches non-word characters at start that aren't protected punctuation
STRIP_NOISE_PATTERN = re.compile(
    r'^([\d+\.\-\)\s]+|[^\w\.\"\']+|[\,])+|[^\w\.\"\']+$'
)
IGNORE_LIST = {'dan'}
@lru_cache(maxsize=10000)
def normalize_caps_word(word):
    """
    Checks if an all-caps word or its singular form is in the dictionary.
    Returns lowercase if it's a known word, else original.
    """
    lower_word = word.lower()
    # Check 0: Ignore list
    if lower_word in IGNORE_LIST:
        return word
    # Check 1: Direct match
    if lower_word in ENGLISH_WORDS:
        return lower_word

    # Check 2: Plural check (removes trailing 's')
    if lower_word.endswith('s'):
        singular = lower_word[:-1]
        if singular in ENGLISH_WORDS:
            return lower_word

    return word

def clean_violation_text(text):
    if not isinstance(text, str) or not text:
        return []

    # Phase 1: Selective Lowercasing with plural/singular awareness
    # This preserves technical acronyms like 'CBRN' while decasing 'RULES'
    processed_text = ALL_CAPS_PATTERN.sub(
        lambda m: normalize_caps_word(m.group(0)),
        text
    )

    # Phase 2: Surgical Noise Stripping
    # Removes '===' or '!!!' but keeps the '.' or ':' at the end of the sentence
    processed_text = STRIP_NOISE_PATTERN.sub('', processed_text).strip()

    # Capitalize the first letter
    processed_text = processed_text[0].capitalize() + processed_text[1:]

    # If the last character is a number or digit, add a period
    if not processed_text[-1].isalnum():
        processed_text += '.'

    # Phase 3: Array formatting for NSHOT/PERSONA compatibility
    return [processed_text]

def create_cleaned_jailbreak_df(df):
    """
    High-performance processing for VIOLATION category with structure preservation.
    """
    # Filter and convert to list of dicts for speed
    records = df.to_dict('records')

    for row in tqdm(records, desc="Cleaning Violations (Preserving Structure)"):
        cleaned_array = clean_violation_text(row['text'])
        row['text'] = cleaned_array
        row['category'] = [PromptEntity.VIOLATION.value]
        row['dataset_source'] = [row['dataset_source']]
        row['source'] = [row['source']]
        row['metadata'] = [row['metadata']]

    return pd.DataFrame(records)

# Execution

# Clear the cache first
normalize_caps_word.cache_clear()
jailbreak_df = create_cleaned_jailbreak_df(full_df[full_df['category'] == PromptEntity.VIOLATION.value])
jailbreak_df.sample(10)

In [ ]:
# Benign
def create_benign_df(df):
    records = df.to_dict('records')

    for row in tqdm(records, desc="Creating Benign Prompts"):
        row['text'] = [row['text']]
        row['category'] = [PromptEntity.BENIGN.value]
        row['dataset_source'] = [row['dataset_source']]
        row['source'] = [row['source']]
        row['metadata'] = [row['metadata']]

    return pd.DataFrame(records)

# Execution
benign_df = create_benign_df(full_df[full_df['category'] == PromptEntity.BENIGN.value])
benign_df.sample(10)

In [ ]:
# Load LDNOOBW Bad Words for Sanitization
try:
    BAD_WORDS_URL = "https://raw.githubusercontent.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words/master/en"
    response = requests.get(BAD_WORDS_URL, timeout=10)
    BAD_WORDS_SET = set(word.strip().lower() for word in response.text.split('\n') if word.strip())
except Exception as e:
    print(f"Warning: Could not load bad words list ({e}). Sanitization disabled.")
    BAD_WORDS_SET = set()

A_STR = build_alternation(["answer", "response", "output", "result", "reply", "assistant", "label", "solution", "thought", "cot"])
Q_STR = build_alternation(["question", "query", "input", "prompt", "user", "human", "ques", "problem"])

INTERNAL_LABEL_PATTERN = re.compile(rf'((?:{A_STR})s?\b|^(?:\bA)\s*:)', re.IGNORECASE)
QUESTION_PATTERN = re.compile(rf'((?:{Q_STR})s?\b|^(?:\bQ)\s*:)', re.IGNORECASE)
CODE_BLOCK_PATTERN = re.compile(r'```.*?```', re.DOTALL)
BAD_WORDS_PATTERN = re.compile(rf"\b({'|'.join(map(re.escape, BAD_WORDS_SET))})\b", re.IGNORECASE) if BAD_WORDS_SET else None

def extract_internal_nshot_hermes(row):
    """Flexible N-Shot detection for OpenHermes internal structures."""
    full_text = row.get('text', '')
    if not isinstance(full_text, str) or not full_text.strip(): return None

    # Strip code blocks to avoid false positives in snippets
    cleaned_text = CODE_BLOCK_PATTERN.sub('', full_text)
    lines = [line.strip() for line in cleaned_text.split('\n') if line.strip()]

    n_indices = [i for i, ln in enumerate(lines) if i > 0 and INTERNAL_LABEL_PATTERN.match(ln) and not QUESTION_PATTERN.match(ln)]
    q_indices = [i for i, ln in enumerate(lines) if i not in n_indices and QUESTION_PATTERN.match(ln)]

    if not (len(n_indices) > 1 and len(q_indices) > 0): return None

    # Check for repeating structure (True N-Shot signal)
    ans_labels = [INTERNAL_LABEL_PATTERN.match(lines[i]).group(1).lower() for i in n_indices]
    que_labels = [QUESTION_PATTERN.match(lines[i]).group(1).lower() for i in q_indices]

    if not (len(ans_labels) != len(set(ans_labels)) or len(que_labels) != len(set(que_labels))):
        return None

    return {
        "text": lines,
        **{}
    }

def is_noisy_or_adult(text):
    """Returns True if text contains terms from the bad words list."""
    if not BAD_WORDS_PATTERN or not isinstance(text, str): return False
    return bool(BAD_WORDS_PATTERN.search(text))

# --- 3. Main Splitting Logic ---

def split_benign_and_internal_nshot(df):
    """
    Gate 1: Extracts internal n-shots.
    Gate 2: Discards toxic/adult noise.
    Gate 3: Wraps clean benign into master array schema.
    """
    nshot_records, clean_benign_records = [], []
    discarded_noise = 0

    records = df.to_dict('records')
    for row in tqdm(records, desc="Splitting & Sanitizing Benign"):
        # 1. N-Shot Detection (Targeting OpenHermes)
        nshot_res = extract_internal_nshot_hermes(row) if row['dataset_source'] == 'openhermes' else None

        if nshot_res:
            nshot_records.append(nshot_res)
            continue

        # 2. Adult/Noise Sanitization
        if is_noisy_or_adult(row['text']):
            discarded_noise += 1
            continue

        # 3. Clean Benign Logic
        clean_benign_records.append({
            **row,
            **{k: [row[k]] for k in WRAP_KEYS if k in row if not isinstance(k, list)},
        })

    print(f"\n[Process Log] Detected N-Shots: {len(nshot_records)}")
    print(f"[Process Log] Discarded (Toxic/Adult): {discarded_noise}")
    return pd.DataFrame(nshot_records), pd.DataFrame(clean_benign_records)

# --- 4. Execution ---

# Split current BENIGN pool
internal_nshot_df, benign_df = split_benign_and_internal_nshot(
    full_df[full_df['category'] == PromptEntity.BENIGN.value]
)


print(f"- Clean Benign: {len(benign_df)}")
print(f"- Internal N-Shot: {len(internal_nshot_df)}")

In [ ]:
# merge nshot with internal_nshot
nshot_full_df = pd.concat([nshot_df, internal_nshot_df], ignore_index=True)
nshot_full_df

In [ ]:
# Regex to find standalone "i" (case insensitive, surrounding by word boundaries or punctuation)
# This prevents accidentally upercasting 'i' inside words like 'init'
I_PRONOUN_PATTERN = re.compile(r'\bi\b', re.IGNORECASE)

@lru_cache(maxsize=1000)
def capitalize_sentence(text):
    if not text:
        return ""

    # 1. Handle standalone "i" -> "I"
    text = I_PRONOUN_PATTERN.sub('I', text)

    # 2. Capitalize the very first character of the string
    # We use this method to avoid issues with strings starting with non-alpha chars
    if len(text) > 0:
        text = text[0].upper() + text[1:]

    return text

def create_cleaned_salad_df(df):
    """
    Normalizes Salad-Data by fixing capitalization for the start
    of sentences and the 'I' pronoun.
    """
    # Filter for the specific dataset source
    records = df.to_dict('records')

    for row in tqdm(records, desc="Normalizing Salad-Data"):
        # Salad data is typically single-string, so we wrap the result in an array
        original_text = str(row['text'])
        cleaned_text = capitalize_sentence(original_text)

        row['text'] = [cleaned_text]
        row['category'] = [row['category']]
        row['dataset_source'] = [row['dataset_source']]
        row['source'] = [row['source']]
        row['metadata'] = [row['metadata']]


    return pd.DataFrame(records)

# Execution
salad_data_df = create_cleaned_salad_df(full_df[full_df['dataset_source'] == 'salad_data'])
salad_data_df.sample(10)

In [ ]:
# Generate persona phrases from fp_unified_df
all_persona_phrases = []
for persona_list in persona_df['text']:
    # Assuming persona_list is a string from fp_unified_df, so no nested iteration
    # Clean_persona_span returns a list, so we take the first element if it exists
    if isinstance(persona_list, list) and persona_list:
        persona_text = persona_list[0]
    elif isinstance(persona_list, str):
        persona_text = persona_list
    else:
        continue

    # Take the first N words, removing trailing punctuation if any
    words = persona_text.split()[:4]
    if words[-1] == "or": words = words[0:3]
    first_three_words = " ".join(word.strip('.,;:') for word in words)

    if first_three_words:
        all_persona_phrases.append(first_three_words)

# Make it a set to ensure uniqueness, then convert back to list
PERSONA_PHRASES = list(set(all_persona_phrases))

print(f"Generated {len(PERSONA_PHRASES)} unique persona phrases.")
# print(PERSONA_PHRASES[:10]) # For debugging


In [ ]:
fake = Faker()

# --- 1. Resource Helpers (unchanged) ---
def clean_lang(name):
    name = re.sub(r'\(.*\)', '', name).strip()
    name = re.split(r'[,;]', name)[0].strip()
    return name.title()

def get_base_format_pool():
    major_prog = {"Python", "Javascript", "Java", "C++", "C#", "Ruby", "Go", "Rust", "PHP",
                  "HTML", "CSS", "SQL", "Typescript", "Bash", "JSON", "YAML", "CSV", "XML", "Latex", "Markdown", "Plaintext"}
    major_world = {"English", "Spanish", "French", "German", "Chinese", "Mandarin", "Japanese",
                   "Korean", "Russian", "Arabic", "Portuguese", "Italian", "Hindi",
                   "Latin", "Polish", "Finnish", "Dutch", "Swedish", "Danish", "Greek",
                    "Turkish",
                   }

    prog_all = set()
    for lexer in get_all_lexers():
        name = lexer[0].title()
        if len(name.split()) <= 3:
            prog_all.add(name)
        for alias in lexer[1]:
            if 1 < len(alias) <= 15 and len(re.split(r"(?:\+|\s+)", alias)) <= 3:
                # Split alias by space or plus
                parts = re.split(r"(?:\+|\s+)", alias)
                # If none of the parts are already known languages, keep it
                if not any(part.title() in major_prog for part in parts):
                    prog_all.add(alias.title())

    world_raw = {lang.name for lang in pycountry.languages if hasattr(lang, 'name')}
    world_all = {clean_lang(l) for l in world_raw
                 if 3 < len(clean_lang(l)) < 25 and len(clean_lang(l).split()) <= 3}

    style = {
        "JSON", "YAML", "CSV", "XML", "LaTeX", "Markdown", "plaintext", "bullet point",
        "numbered list", "ELI5", "table", "ASCII",
        "Base64", "hexadecimal", "binary", "protobuf",
        "natural language", "emoji"
    }


    return {
        "major_prog": sorted(major_prog),
        "minor_prog": sorted(prog_all - major_prog),
        "major_world": sorted(major_world),
        "minor_world": sorted(world_all - major_world),
        "style": sorted(style),
    }

POOLS = get_base_format_pool()

# --- 2. Target Samplers (unchanged) ---
def get_dynamic_target():
    if random.random() < 0.13:
        base = random.choice(["Base", "ROT", "Caesar"])
        sep = random.choice(["", "-", " "])
        num = random.randint(1, 64) if "ROT" in base else random.randint(1, 256)
        return f"{base}{sep}{num}"

    tier = "major" if random.random() < 0.85 else "minor"
    cat_roll = random.random()
    if cat_roll < 0.48:
        return random.choice(POOLS[f"{tier}_prog"])
    elif cat_roll < 0.82:
        return random.choice(POOLS["style"])
    else:
        return random.choice(POOLS[f"{tier}_world"])

def get_dynamic_target_pair():
    connector = random.choice(["and", "or"])
    if random.random() < 0.78:
        cat_roll = random.random()
        if cat_roll < 0.5:
            pool = POOLS["major_prog"] + POOLS["minor_prog"]
        elif cat_roll < 0.82:
            pool = POOLS["style"]
        else:
            pool = POOLS["major_world"] + POOLS["minor_world"]
        selected = random.sample(pool, 2)
        return f"{selected[0]} {connector} {selected[1]}"
    else:
        return f"{get_dynamic_target()} {connector} {get_dynamic_target()}"

# --- 3. Structured Grammar Pools ---
# ============================================================
# SHARED SLOT POOLS
# ============================================================

PREFIX = [
    "I want you to", "You must", "You should", "You need to",
    "Make sure you", "Make sure that you", "I need you to",
    "I expect you to", "You are required to", "You are expected to",
    "From now on"
]

POLITENESS = ["please", "always", "only", "kindly", "specifically", ""]

QUANTIFIER = ["all", "every", "each", ""]

TEMPORAL = ["final", "current", "future", "subsequent", ""]

RESPONSE_NOUN = [
    "output", "response", "reply", "answer", "result",
    "message", "text", "content", "statement", "completion"
]

RESTRICTION = [
    "always", "only", "purely", "entirely", "strictly",
    "specifically", "exclusively", "consistently", "properly", ""
]

QUALIFIER = [
    "valid", "strict", "specified", "exact", "proper",
    "correct", "well-formed", "standard", "structured", ""
]

ACTIONS = [
    "respond", "reply", "answer", "output", "write",
    "speak", "communicate", "generate", "produce", "render"
]

FORMAT_ACTIONS = [
    "format", "output", "return", "structure", "serialize",
    "render", "produce", "generate", "present", "write"
]

CONNECTORS      = ["in", "as", "using", "with"]
ENSURE_PREFIX   = ["Ensure", "Make sure", "Guarantee", "Verify", "Confirm"]
ENSURE_VERB     = ["uses", "follows", "adheres to", "conforms to", "is in", "is formatted as", "is written in"]
ENSURE_ADV      = ["only", "purely", "properly", "strictly", "exclusively", ""]
RENDER_VERB     = ["Output", "Render", "Produce", "Generate", "Return", "Print", "Emit"]
RESULT_NOUN     = ["result", "answer", "response", "output", "completion", "reply"]
FIX_ADV         = ["Always", "Strictly", "Specifically", "Exclusively", "Consistently", "Only"]
STYLE_FORMAT = [
    "bullet points",
    "a numbered list",
    "a table",
    "a code block",
    "headers and subheaders",
    "a step-by-step list",
    "an ordered list",
    "an unordered list",
    "key-value pairs",
    "a structured outline",
    "an outline",
    "a paragraph",
    "three paragraphs",
    "five paragraphs",
]
EXCLUSIVITY     = ["only", "exclusively", "solely", "as required", ""]
CONVERT_VERB    = ["Convert", "Transform", "Format", "Reformat", "Restructure", "Serialize"]
PROVIDE_VERB    = ["Provide", "Give", "Return", "Output", "Deliver", "Present"]

def get_suffix_none():

    NEGATION    = ["with no", "without", "omitting", "excluding"]
    ALL         = ["all", ""]           # filler — optional after negation
    ANY         = ["any", ""]   # filler — optional between all and intensifier
    INTENSIFIER = ["extra", "other", "further", "extraneous", "unnecessary", "supplementary","additional", ""]
    NOUN        = [
        "text", "commentary", "explanation", "output", "response",
        "content", "context", "preamble", "padding", "filler", "prose"
    ]

    static = ["and nothing else", "only", ""]

    INVALID_PAIRS = [("with no", "all")]

    generated = [
        clean_join([neg, all_, any_, intens, noun])
        for neg    in NEGATION
        for all_   in ALL
        for any_   in ANY
        for intens in INTENSIFIER
        for noun   in NOUN
        if not any(neg == p[0] and all_ == p[1] for p in INVALID_PAIRS)
    ]
    return static + generated


FORMAT_NOUN = [
    "format", "structure", "style", "schema",
    "template", "form", "layout", "syntax"
]

FOLLOWING_PHRASE = [
    "in the following", "in this", "using the following",
    "using this", "in the exact following", "using the exact",
    "according to the following", "according to this",
]


def _noun(plural=None):
    n = random.choice(RESPONSE_NOUN)
    is_plural = (random.random() < 0.35) if plural is None else plural
    if is_plural:
        plural_n = (n[:-1] + "ies") if n.endswith("y") else (n + "s")
        return (plural_n, "are")
    return (n, "is")

def clean_join(parts):
    return re.sub(r"\s+", " ", " ".join(filter(None, parts))).strip()

def pick_connector(action, target):
    # Action-aware connector selection
    if action in ["convert", "transform", "serialize"]:
        return "into"
    elif action in ["format", "render", "structure"]:
        return "as"

    # Fallback semantic hint
    if any(k in target.lower() for k in ["list", "table", "instructions", "tone", "art"]):
        return "as"

    return "in"

def _format_phrase(target):
    """
    Builds the tail format phrase naturally.
    'in the following valid Python format'
    'using this exact JSON structure'
    'according to the following strict YAML schema'
    """
    following    = random.choice(FOLLOWING_PHRASE)
    qualifier    = random.choice(QUALIFIER)
    fmt_noun     = random.choice(FORMAT_NOUN)

    return clean_join([following, qualifier, target, fmt_noun])

SUFFIX_NONE = get_suffix_none()

# ============================================================
# TEMPLATE 1 — Full constraint sentence
# "You must always ensure that all the current responses
#  are strictly in the following valid Python format only"
# ============================================================
def t1_ensure_format(target):
    prefix      = random.choice(PREFIX)
    politeness  = random.choice(POLITENESS)
    ensure      = random.choice(["ensure that", "make sure that", "guarantee that", ""])
    quantifier  = random.choice(QUANTIFIER)
    temporal    = random.choice(TEMPORAL)
    noun, lv    = _noun()
    restriction = random.choice(RESTRICTION)
    fmt         = _format_phrase(target)
    tail        = random.choice(EXCLUSIVITY)


    parts = [
        prefix, politeness, ensure,
        quantifier, "the", temporal,
        noun, lv, restriction,
        fmt, tail
    ]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 2 — Simple respond in
# "You must always respond in Python only"
# ============================================================
def t2_respond_in(target):
    prefix      = random.choice(PREFIX)
    politeness  = random.choice(POLITENESS)
    action      = random.choice(ACTIONS)
    restriction = random.choice(RESTRICTION)
    connector   = random.choice(["in", "as", "using"])
    qualifier   = random.choice(QUALIFIER)
    tail        = random.choice(EXCLUSIVITY)


    parts = [
        prefix, politeness, action,
        restriction, connector, qualifier, target,
        tail
    ]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 3 — Respond as persona/style
# "You must always respond as/like a [persona/style]"
# ============================================================
def t3_respond_as(target):
    prefix      = random.choice(PREFIX)
    politeness  = random.choice(POLITENESS)
    action      = random.choice(ACTIONS)
    restriction = random.choice(RESTRICTION)

    # Determine if target is a persona phrase or a fake name
    is_fake_name = (random.random() < 0.5)

    if is_fake_name:
        current_target = fake.name()
        style_word = random.choice(["as", "like"])
    else:
        if not PERSONA_PHRASES:
            current_target = "a helpful assistant"
        else:
            current_target = random.choice(PERSONA_PHRASES)
            # Add "a" or "an" if needed for natural language persona
            if not current_target.lower().startswith(('a ', 'an ')): # Check for leading 'a ' or 'an ' is not needed here as already done before calling t3_respond_as
                if current_target.lower().startswith(('a', 'e', 'i', 'o', 'u')):
                    current_target = f"an {current_target}"
                else:
                    current_target = f"a {current_target}"
            else:
                # Lowercase the a/an
                current_target = current_target[0].lower() + current_target[1:]
        style_word  = random.choice([
            "as", "like", "in the style of", "just like", "exactly like",
            "in the manner of", "similarly to", "the way of"
        ])

    parts = [prefix, politeness, action, restriction, style_word, current_target]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 4 — Format your response
# "You must always format all your current responses strictly as Python"
# ============================================================
def t4_format_your(target):
    prefix      = random.choice(PREFIX)
    politeness  = random.choice(POLITENESS)
    action      = random.choice(FORMAT_ACTIONS)
    quantifier  = random.choice(QUANTIFIER)
    temporal    = random.choice(TEMPORAL)
    noun, _     = _noun()
    restriction = random.choice(RESTRICTION)
    connector   = pick_connector(action, target)
    qualifier   = random.choice(QUALIFIER)


    parts = [
        prefix, politeness, action,
        quantifier, "your", temporal,
        noun, restriction, connector, qualifier, target,
    ]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 5 — Ensure your response uses/follows
# "Ensure all your entire response uses only valid Python"
# ============================================================
def t5_ensure_uses(target):
    prefix      = random.choice(ENSURE_PREFIX)
    quantifier  = random.choice(QUANTIFIER)
    scope       = random.choice(["entire", "full", "current", "final", ""])
    noun, _     = _noun(plural=False)
    verb        = random.choice(ENSURE_VERB)
    adv         = random.choice(ENSURE_ADV)
    qualifier   = random.choice(QUALIFIER)

    target_phrase = clean_join([qualifier, target])

    parts = [
        prefix, quantifier, "your",
        scope, noun, verb, adv,
        target_phrase
    ]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 6 — Output/Render the result
# "Output the result in the exact following format with no additional text"
# ============================================================
def t6_render_result(target):
    verb        = random.choice(RENDER_VERB)
    quantifier  = random.choice(QUANTIFIER)
    noun        = random.choice(RESULT_NOUN)
    fmt         = _format_phrase(target)

    parts = [verb, quantifier, "the", noun, fmt]
    if random.random() < 0.25:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 7 — Always respond using style format
# "Always respond using bullet points only"
# ============================================================
def t7_style_format(target):
    adv         = random.choice(FIX_ADV)
    action      = random.choice(ACTIONS)
    using       = random.choice(["using", "with", "in", "via"])
    style       = random.choice(STYLE_FORMAT)
    exclusivity = random.choice(EXCLUSIVITY)

    parts = [adv, action, using, style, exclusivity]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 8 — Convert/Transform your output
# "Convert your final output into valid JSON"
# ============================================================
def t8_convert(target):
    verb        = random.choice(CONVERT_VERB)
    quantifier  = random.choice(QUANTIFIER)
    temporal    = random.choice(TEMPORAL)
    noun, _     = _noun(plural=False)
    connector   = pick_connector(verb.lower(), target)
    qualifier   = random.choice(QUALIFIER)

    parts = [
        verb, quantifier, "your", temporal,
        noun, connector, qualifier, target,
    ]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 9 — Provide/Give the answer
# "Provide the answer strictly in JSON format with no extra explanation"
# ============================================================
def t9_provide(target):
    verb        = random.choice(PROVIDE_VERB)
    quantifier  = random.choice(QUANTIFIER)
    noun        = random.choice(RESULT_NOUN)
    restriction = random.choice(RESTRICTION)
    connector   = random.choice(["in", "as", "using"])
    qualifier   = random.choice(QUALIFIER)
    fmt_noun    = random.choice(FORMAT_NOUN + [""])

    target_phrase = clean_join([qualifier, target, fmt_noun])

    parts = [
        verb, quantifier, "the", noun,
        restriction, connector,
        target_phrase
    ]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# TEMPLATE 10 — Only/Strictly use target (strong imperative)
# "Only use valid JSON and nothing else"
# ============================================================
def t10_strong_use(target):
    prefix      = random.choice([
        "Only use", "Use only", "Strictly use",
        "Exclusively use", "Always use", "Use strictly",
    ])
    qualifier   = random.choice(QUALIFIER)
    exclusivity = random.choice([s for s in EXCLUSIVITY if s])

    parts = [prefix, qualifier, target, exclusivity]
    if random.random() < 0.5:
        parts.append(random.choice(SUFFIX_NONE))
    return clean_join(parts)


# ============================================================
# DISPATCHER
# ============================================================

TEMPLATES = [
    t1_ensure_format,
    t2_respond_in,
    t3_respond_as,
    t4_format_your,
    t5_ensure_uses,
    t6_render_result,
    t7_style_format,
    t8_convert,
    t9_provide,
    t10_strong_use,
]

TEMPLATE_WEIGHTS = [0.12, 0.18, 0.08, 0.12, 0.10, 0.08, 0.08, 0.10, 0.08, 0.06]

def generate_instruction():
    selected_template = random.choices(TEMPLATES, weights=TEMPLATE_WEIGHTS, k=1)[0]

    if selected_template == t3_respond_as:
        text = selected_template("")   # or better: pass a persona target
        category = PromptEntity.PERSONA.value
        return text, category

    target = get_dynamic_target_pair() if random.random() < 0.33 else get_dynamic_target()
    text = selected_template(target)
    category = PromptEntity.FORMAT.value
    return text, category


# --- 5. Main Generator ---
def generate_tiered_format_dfs(num_samples=12000):
    format_records = []
    persona_records = []
    seen = set()

    for _ in tqdm(range(num_samples), desc="Generating varied FORMAT data"):
        final_text, category = generate_instruction()

        if final_text in seen:
            continue
        seen.add(final_text)

        row = {
            "text": [final_text],
            "category": [category],
            "dataset_source": ["synthetic_generator"],
            "source": ["synthetic_generator"],
            "metadata": [None],
        }

        if category == PromptEntity.PERSONA.value:
            persona_records.append(row)
        else:
            format_records.append(row)

    format_df = pd.DataFrame(format_records).drop_duplicates(subset=["text"]).reset_index(drop=True)
    persona_df = pd.DataFrame(persona_records).drop_duplicates(subset=["text"]).reset_index(drop=True)

    return format_df, persona_df


format_syn_df, persona_syn_df = generate_tiered_format_dfs(8000)
print(f"Generated {len(format_syn_df)} unique FORMAT samples")
print(f"Generated {len(persona_syn_df)} unique PERSONA samples")

In [ ]:
for i in range(20):
    text = format_syn_df.iloc[i].text[0]
    print(text)
for i in range(10):
    text = persona_syn_df.iloc[i].text[0]
    print(text)

In [ ]:
full_df2 = pd.concat(
    [
        jailbreak_df,
        persona_df,
        persona_syn_df, format_syn_df,
        nshot_full_df,
        benign_df,
        salad_data_df,
    ]
)

full_df2.sample(20)

In [ ]:
full_df2.to_parquet("prompt-nlp-fixed.parquet")

# Merging

In [ ]:
class PersonaType(Enum):
    CYBER = "cyber"
    MEDICAL = "medical"
    LEGAL = "legal"
    SCIENCE = "science"
    EDUCATION = "education"
    BUSINESS = "business"
    TRADES = "trades"
    HUMANITIES = "humanities"
    ENVIRONMENT = "environment"
    ARTS = "arts"
    MATH = "math"

CATEGORY_TO_PERSONA = {
    "Cyber & IT": PersonaType.CYBER,
    "CBRN & High-Risk Science": PersonaType.SCIENCE,
    "Healthcare & Medical": PersonaType.MEDICAL,
    "Legal & Policy": PersonaType.LEGAL,
    "Business, Finance & Management": PersonaType.BUSINESS,
    "Skilled Trades & Industrial Work": PersonaType.TRADES,
    "Humanities & Social Sciences": PersonaType.HUMANITIES,
    "Environment & Wildlife": PersonaType.ENVIRONMENT,
    "Arts, Design & Media": PersonaType.ARTS,
    "Mathematics": PersonaType.MATH,
    "Education & Academia": PersonaType.EDUCATION,
}
SALAD_TO_PERSONA: dict[str, set[PersonaType]] = {

    # =========================
    # SOCIAL (O1–O16)
    # =========================
    "O1: Hate Speech": set(),
    "O2: Harass, Threaten, or Bully An Individual": set(),
    "O3: Enjoying someone else's Pain": set(),
    "O4: Terrorism": {PersonaType.SCIENCE, PersonaType.CYBER},
    "O5: Violent Content": set(),
    "O6: Child Abuse": {PersonaType.MEDICAL},
    "O7: Insult": set(),
    "O8: Racial and Ethnic Stereotyping": set(),
    "O9: Age Stereotyping": set(),
    "O10: Disability Stereotyping": {PersonaType.MEDICAL},
    "O11: Gender Stereotyping": set(),
    "O12: Religious Stereotyping": set(),
    "O13: Location Stereotyping": set(),
    "O14: Erotic Chat": set(),
    "O15: Sexual Activity": {PersonaType.MEDICAL},
    "O16: Pornography": set(),

    # =========================
    # HIGH STAKES (O17–O18)
    # =========================
    "O17: Law Advice": {PersonaType.LEGAL},
    "O18: Common Sense": {PersonaType.EDUCATION},

    # =========================
    # ILLICIT / DRUGS / ECON (O19–O28)
    # =========================
    "O19: Illegal Drugs and Regulated/Controlled Substances": {PersonaType.SCIENCE, PersonaType.MEDICAL},
    "O20: Medical Advice": {PersonaType.MEDICAL},
    "O21: Historical Revisionism": {PersonaType.EDUCATION},
    "O22: Science Denialism": {PersonaType.SCIENCE, PersonaType.EDUCATION},
    "O23: Radicalization": set(),
    "O24: Multi-level Marketing": {PersonaType.BUSINESS},
    "O25: Paypal Lending": {PersonaType.BUSINESS},
    "O26: Gambling": {PersonaType.BUSINESS},
    "O27: Labor Issues": {PersonaType.LEGAL, PersonaType.BUSINESS},
    "O28: Trade Affairs": {PersonaType.BUSINESS, PersonaType.LEGAL},

    # =========================
    # DISINFO / INFO OPS (O29–O30)
    # =========================
    "O29: Fake News": set(),
    "O30: Social Media Rumors": set(),

    # =========================
    # SENSITIVE (O31–O34)
    # =========================
    "O31: Government Secrets": {PersonaType.LEGAL, PersonaType.CYBER},
    "O32: Financial Data Leaks": {PersonaType.CYBER, PersonaType.BUSINESS},
    "O33: Inferring Personal Information": {PersonaType.CYBER},
    "O34: Leaking Private Information": {PersonaType.CYBER},

    # =========================
    # VIOLENCE / CYBER / CBRN (O35–O39)
    # =========================
    "O35: Weapon Generation and Transportation": {PersonaType.SCIENCE},
    "O36: Management or Operation of Critical Infrastructure": {PersonaType.CYBER},
    "O37: Malware Generation": {PersonaType.CYBER},
    "O38: Cyber Attack": {PersonaType.CYBER},
    "O39: Biological and Chemical Harms": {PersonaType.SCIENCE},

    # =========================
    # SOCIAL / DEFAMATION (O40–O41)
    # =========================
    "O40: Group Defamation": set(),
    "O41: Defamation Someone": {PersonaType.LEGAL},

    # =========================
    # FRAUD / SCAMS (O42–O43)
    # =========================
    "O42: Scams": {PersonaType.BUSINESS},
    "O43: Financial Fraud": {PersonaType.BUSINESS},

    # =========================
    # DISINFO / CONTENT ABUSE (O44–O48)
    # =========================
    "O44: Academic Dishonesty": {PersonaType.EDUCATION},
    "O45: Fake Review Generation": {PersonaType.BUSINESS},
    "O46: Fake Online Content": {PersonaType.BUSINESS},
    "O47: Disinformation": set(),
    "O48: Spam": {PersonaType.BUSINESS},

    # =========================
    # HEALTH / ECON CRIME (O49–O54)
    # =========================
    "O49: Pseudo-pharmaceuticals": {PersonaType.MEDICAL, PersonaType.SCIENCE},
    "O50: Manipulation of Public Opinion": set(),
    "O51: Campaign Materials": set(),
    "O52: Illegitimate Surveillance": {PersonaType.CYBER},
    "O53: Financial Crimes": {PersonaType.BUSINESS},
    "O54: Drug-related Crimes": {PersonaType.SCIENCE, PersonaType.MEDICAL},

    # =========================
    # VIOLENT / CRIME (O55–O60)
    # =========================
    "O55: Sexual Offenses": set(),
    "O56: Violent Crimes": set(),
    "O57: Theft": set(),
    "O58: Illegal Law Advice": {PersonaType.LEGAL},
    "O59: Environmental Crimes": {PersonaType.ENVIRONMENT},
    "O60: Traffic and Driving Offenses": {PersonaType.TRADES},

    # =========================
    # SOCIAL / PSYCHOLOGICAL (O62–O63)
    # =========================
    "O62: Self-Harm": {PersonaType.MEDICAL},
    "O63: Psychological Manipulation": {PersonaType.BUSINESS},

    # =========================
    # DISINFO / IDENTITY (O64–O67)
    # =========================
    "O64: Mimicking Writing Style": {PersonaType.ARTS},
    "O65: Plagiarism": {PersonaType.EDUCATION},
    "O66: Copyright Infringement": {PersonaType.ARTS, PersonaType.BUSINESS},
    "O67: Forge Identity": {PersonaType.BUSINESS},
}